# Phase 13: CIFAR-100 TickNet-L Large v2 (14 khối) + CutMix α=1,0

TickNet-L gốc của nhóm có 7 khối. Theo cách TickNet-small trong bài báo
tăng từ 5 lên 10 khối, bản v2 tăng thành 14 khối, phân bổ theo stage
1–2–5–5–1. Bản này giữ nguyên độ rộng của TickNet-L Large Phase 12 và
giữ khối PDP cải tiến của nhóm; chỉ mượn ý tưởng tăng gấp đôi số khối,
không sao chép lịch kênh riêng của TickNet-small gốc.
Đây là thí nghiệm đối chứng với Phase 9 α=1,0: **chỉ đổi kiến trúc** từ
TickNet-L sang TickNet-L Large v2. Công thức train giữ SGD lr=0.175,
weight decay=0.0005, RandAugment N=2/M=7, CutMix α=1,0 p=0.5,
batch 128, seed 42, 45.000 train / 5.000 validation, cosine LR **200 epochs**.
Không dùng EMA. TickNet-L Large v2 có 4.545.900 tham số và 0,7888G FLOPs
cho một ảnh 32×32, dưới giới hạn 6M tham số / 1G FLOPs.
Phase 9 α=1,0: Best Val 77,44%, Test Top-1 77,66%. Chọn checkpoint bằng
validation Top-1; test chỉ đánh giá một lần sau khi train đủ 200 epochs.

Model Large v2 có kiến trúc khác nên phải chạy **mới từ epoch 1**. Không resume
checkpoint Phase 9/10/11/12. Nếu cần chia phiên, dùng recovery ZIP Phase 13
với cùng ZIP mã nguồn, runtime và GPU.

Upload `TickNets_CIFAR100_L_LargeV2_CutMix_200_Source.zip` làm Kaggle Dataset
và **Add Input**. Chọn GPU và bật Internet; notebook tự tải CIFAR-100 chính thức.
Nguồn có thể là ZIP hoặc thư mục Kaggle đã giải nén. Không cần gắn CIFAR-100
riêng. File kết quả là
`phase13_cifar100_l_large_v2_cutmix_a10_200ep_results.zip` trong `/kaggle/working`.


In [ ]:
from pathlib import Path

SOURCE_ZIP = None       # Exact path if several source copies are attached
SOURCE_ROOT = None      # Exact extracted project folder if needed
KAGGLE_INPUT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
EPOCHS_PER_SESSION = None  # Example: 25 epochs per session
RESUME_ROOT = None         # Folder containing the named Phase 13 run


In [ ]:
import hashlib, importlib.util, json, os, sys, zipfile
from pathlib import Path, PurePosixPath

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
SOURCE_PATHS = ['configs/final/cifar100_sgd_lr0175_wd0005_ra_cutmix.json', 'configs/final/cifar100_ticknet_l_large_v2_cutmix_a10_200.json', 'download_cifar.py', 'models/cifar_data.py', 'models/cifar_experiment.py', 'models/cifar_training.py', 'models/cifar_training_large_v2.py', 'models/common.py', 'models/datasets.py', 'models/model_profile.py', 'models/reproducibility.py', 'models/SE_Attention.py', 'models/TickNet.py', 'models/ticknet_l.py', 'models/ticknet_l_large_v2.py', 'scripts/aggregate_grid_search.py', 'scripts/download_cifar.py', 'scripts/run_kaggle_cutmix.py', 'scripts/run_kaggle_cutmix_large_v2_200.py', 'scripts/verify_cifar_run.py', 'train_cifar.py', 'train_cifar_cutmix.py', 'train_cifar_cutmix_large_v2.py', 'train_cifar_randaugment.py']
EMBEDDED_SOURCE_B85 = (
    'c-qw4Q=DW?)+qe4-DTUhjV#-?ZFZr%Y}>YNv&*(^`#dveUd)_%zjJQCi}+>iyxJKncH~<B9ibo%3Wf><1O)Y0BT_K<o=>d&tA_#%1jO;zr?I`AnWecigPEnB'
    'p$&tvrJ11<6C<O(v$=`BjT0ji2b;dT2_qvTo4%8wzOk!|t)&OOm9xE_>>o*l0YW(N8DBZ%x@c+?*KRncKFL^2V#czya(9M-;)(BW)A#vDz6QNK!{=-)_QXnm'
    'LSnQyuV6-}EtsCT=C`Ok6-Px`cwvV%I$|J0yBEXG%;4sg`~nl2EoweR1{&z!D0)d;$F`Up49lsud!7BDfNJ6+Z>*71Ek{U_BcS+3_e-_I&{8K}IkquT{-ken'
    'M&xe*rVDJ9>2_QF008qN`?qla)%qup!v8n`eE;(RxL6un+nKuP+vwXEI+>g5yD|Uc=j$6XG3qljGXDR37>=M<P{RJ|wf)tQ|7s@o?shizh9>%d9|3v?Pt`~!'
    'gnmXivFFVGUq!m{0q|u%AnSxPpri+*GiT!0=cO2LMoON)UJvwDKeJXO^zt5_kh(FgFJ(QVWa5sYsOJpk6{J)E`3mMmbLILb`z61?&838NPwg1s4~f@upRD4g'
    'zE-ccgKLsO>zERO;zXiD8D$NS5u+_?Af7Ju>ATC|A@PQ1eg5UXlc0DUy#ETFDi{zD`d{DJ+MAf#I5Q}V=?l5InA*8m+S~ng^G(Wp_5*}So1Zk8c}|iE06c2b'
    '@_72wxl$J4Co-Y_9T2(7kGJd}>$fT8w5GcVS6e(=c+?YBu%sF(xFoL5%vf6QUVTwSb3crOOMLWGNhz%ht3lWm6D>|>XPChlQAzl1wJZ$mk^7SRvoET6q*Ig-'
    'y$BNOyUUV;6gWks{FEGu$)3+Z9)1rUc>cjYcrIXtcnE=G45R<WObw+tK)@(R+JZ|!BBG^pjwZL)oZjPh;u{xKfESpmLKxYSS_$2CYydRvBBxf0u-zvQOio#5'
    'iKAF0j)49g&175w2atd?ebQmhfIr9I_vw3j7kYv21SUD}dm!Lj?W?0L1=SH8od-@@b=VNItI{h>Ok}e3>ge5CI*QM`3{s}5EUp>ZTYiK=#n4Pwn?ZQ}B;$|6'
    'u5R*h9j%nIq{z7KU(f`sIVMMnq)8{hoLY{FCrD-6^-j55Eq${jc078Zzq71hY5SNyr+iEe<G-X7)rwU(V0a0~QM0~*|8uw|hH!{?{|Z&ykN-W*{t+&fzn(5R'
    'Q<uL*N`2j~kQ2%Gn5NP$3WNs=JU&b<&SO-L*&atM!83yR0K$C)CRLCAbW8WL#~v%l(a{pc5<(2mA*mvILUqJ6`|>`Y7`&m?tKw^`VQx;}6S>Dj7_Ntp-osRL'
    'op?GF<5)8pcCLJ|&Er1CuTk>;Wsyu@Su5FRLUUJYZ0MF)3PH*vq1dYu%i@x6ON>KxNOZ`%qDQliRkYhT2H%b#vAt^cBaOm!JkvU&w?V5a{kxcY^$7g6Z-3ca'
    '%@aUlsEs1##HDDGLc7^M#pu-<x`#mxCOrXdsi$5tXsUWsIIsk_;H|b2UDCzf1@+;L2$}PXGqO2=kT>cGjFZ*baxed|-DFn$*S3XLy(unh^$SiC5BAn^Abhfq'
    'w}m<{0MQ$}9maA++zqY`-uULeKNKHTx$&KV`JSnygSh6|^du(!@SQ;_`FA5euZJ|FI2#n>A}l&KkkG(OzZE(542wYB-acL^!L>~%!&g#41zzcnV1y_d4;OZl'
    'W^FtLn`~ddA|gK7O*8#v!Jp>3k0%EuxL2sVTTTPE;8*k!T2lTz;<QjzPJ~IeHS*J40ROqFC6PtFwek;HmcSX))XgZaYiEVpa(b}ATC&Sz4~&;~E-LK&V3**>'
    'WkD>X5>0K;PblXI?g(r+8|sjZwt6=?GOwD#Clg`QxX+TyrzSThI)+yP+Qy)rIkKC58A?eMISa7@_^LgOI1J5~f>HsewMe?&>S#y_F`Bxvz$AP0WP6K`cHneK'
    '@UG*SiyE!$Q&w;&EnnF}0_LZAw0=yOKTZeJ1(GPHvQDcsY|@n*_yYuU!+a}HDB8ut$%>;Y!4`8uCnhu4URipCPNgj1I{DpZ$#Eeg_mp@%dU)hS5SnT(BVPEp'
    'N@p|WYKcutlTjXf&Gug87{7!fHH2u2rwz@Ke7H>A5@cmh%aZ|xB}b-f$Pi2pB|7;7g+6EKH?+#du0*T1ox5DQ(|s(4$Z)w)iE3Y;_J_bHk!o({4=lHM81A{('
    '>l%$)hVOfkK)atJ3EOFJQ}K~#E=*P_Gd>6&1{2`2j@kas-@L83;C5T~_Jwo-hs9tLRtVwN?6DPrMUUf-Tfi16M>w@j)AtJp0U5up`fxiu*ZKURR)7>PfB^%o'
    '-G<oLL%6&J=*GU=eK&88we$MnNBc&&x_P%(8Y0xe4=eM5W~L(*cLuB>McImy)^opMkOOkDoWo!;D<yGNs4n|qq8)UFcW>hd(qzRZZHi1p-)VkP1IVqgHp#ox'
    '$5}XUNB|yb&5G8eJdEAmW#r=~>-mrDnIKV#q6cVkEfIAkbuGmsx%qQ;g~%zHBu2VhlWi~49BJ*h=SDoMXixYuj@uel%huMPrwNfj(nS(spy35Jq$qvGN0unA'
    'RVk}bWjx)J4-L!(mR=A|<1LK}?rrF2FDpP67y*modKpmD?2QMvnmvLd-=|}Uem~uEs_Aznx*+tccb+2%cm?EFlswRF+ScM>-E^E+_-y%$hN68pI(b<6xmLVw'
    '@;z;x0PA$C7y+<Ctf$~P=kZSOD#{d@V8;V!5b1)CB_FK1ABl*Y4B2-msZmQ|Q@Hz35S+sr5YB;2PwX*cqHe(@+E2TIa%}Pw4d(3?t7&WN`7v&vyFTCBhKlBL'
    '@7r}>kpH|MKFI`2&cOfyfs+COq5ju;`1e0ueG@|$!@riqAx&HRb#auhYW+G@f7n&#%0f;zISh0_)&Vx5Y{c57FA`WFxjG61X^O5=^P<j|xA~JV)tH;pkgk8~'
    '!PUjZ1<Q5Ct;YxTd4oGG4JC!9a%{G_Y>QFn71RgWvHf=E$KjX@@zKM`FVx!70$+@nk^R~5hV;Tyt95%WS!Jaqx_!P%3yqvdGVAtPKoszO;G&aQ#$!f#=iGt7'
    'KuP9_yX|V_<4XneHOXD|#m==nUHwFPsE*xwICk??vz<HXIbi-QFZ)*K_WkveXX?T>tEy}b9D2`IUrjf1gy|iNm-rT8#06a^1rLyG$p#k(x(-kp$!yL<Tkq_o'
    'uBhKWCBrKcld7TGe#GH4C55!Z{1$w*cnH;b08$)=odas9r2a;NipN#>mG(tn6*;)>694t`cKF`*%c8Da17fr-hD;K`7$*P4&ld*8n=@^;=jrrD%gRenufI8M'
    '<mSZQSJaSDGCg?j;)NZ}8WRF@XXC`46quPUDMUUkGI3xS|5q0}J8S%S^;AU~1XoW_Z}K-8U3)goxhC#0)@I!{iu(sA@r`|EN=2kR#Xc9b&IQR{E~Yqsl{9be'
    '(m{@R!E?`7YIcver{nEfLWj*;P=`Fldq%H*pOwl?5)*kkb%Jb0Wijwq+7AP~c#7>^XZn=`v?*+@UO52LfX6C?p-vsn_%3v^t|=G%u>3GGR!mla3IY`Q;rI;|'
    '4hp6viNv_VRmeBz?~~aRGNV*;Q;9Kq&8DInE^}UusGqj{tQW;`p_b1i1d5Y2FC{-?eFiWqJuBTjWO_9pMh=(&?AV8@ORfh#>kYO9wG}_-JJpQ@Y2U>LOROZ$'
    'D&GbD18akc6Ve@}+P7P<cnKX2abU7T{jv=i{d0xA)?5TJJM(*PQOv%@=^J#11g#s!9%v&ajABxJHdjSBPJ>kiar})2xIBBRY6#&Pl2-H0NhOGGzQbo|w?{XP'
    '2J-W#+P5tbQgC8X79zI`>Fg~Rf14H2_*Kf&4hiE(&ev1Dv%+(0RT0yz<3=IS0`scanL^0nT?W(>Sp^A=i4jo|p*QHof>6F?GK_)=fR`Px7pjbS3Kb{63Tb0X'
    '!v}7DbklK(k;{rKBhl4RF&9`{Jr6Oi)>l7}AF7RO-4bN;EJ>qut>mcPky$lG=w+)W3Ds0CJ0}^@|2Bj|bcW=;@#nP*uP4_hJ17%`BMRuNmlj(IP>E}X6GiNY'
    '1QD^<X0+wxnS~pDMf_GWJInO?5r@NxXt!irpG==UnH=}drHie#i{>wb6Qu1T<ZnFy)iZ=IZ@xP^W0Wm5`$)u6ZMhT3!`UxapV$=OvBSlYC*k59dosMLP@67Q'
    '-x(O%L?X{y<oiz7CyEgVe7-rU{Sei1Vn}PZ>1!WD*N#P-?iqpKczx4b8aHIez1{b4>Eg#T@WB{HK_YKn1|t_%FBqv)Mwt$hiLjpy4(??d=!IdAtnuiYR)Ktr'
    'J#3&1<dg0S92q>Y0&jd^?JkhyvlMG53MXfqgbEmOjE@@?VUF^pBUp9QoiENwpc$r){fxIS%N+Y1Mvw&viKE7rd}di6Iffgri%##gAE!^#)Kcs^Y&`F^KN2ne'
    '?fhqLhYD!Z3|wQIn+te7MOlDR@2fAJzhcXundDCIcpYGhtw;$A%>t#e*IW8AV7dAkOSv42mZx&TM8U&0)$gPo?Xt93i}m<W(HMA#l}yjO?zqG0CU~E|RGB-L'
    'I<E?@Xj&PbUXba-8@PXvA|pP)1At2g&X2{UM0vTc)M~#~Sqlh#&z6k69vwvFhsor$)?PymD2G+am6BPK=4Ic_zP!#*%jJ}w+c#08c<#OPjHR`$axTM|+cWY8'
    '4@N8P9$w;e!cF*HU53QnTSzVTHfW=i<hiO%Rm&cig|<LakIbtkML5kR&h5gKeVMB}&M68`Nncx|ZJ-%3WD6`bOu*7Xc1^U;=_vvxG22WQ*d>s+V%(Wv|JAq#'
    '-N0#sRyr+2WPq5i!L&gFvRz-&B>Os470OAxF#6Ru^<3}e4!cz;We8vE{#&az)_DAp8+?=%KHDV^LyOuIAhQ>Xq6<W8dr}W3H47ENOUQ~+8YR3O2EkY=43D<k'
    '5qof=OdyS<^++GPqYNVotb004--H4H!6;HFLI+?etW`=`<nb`h@T=Z4cj4ADGa;lgpjGFMYjb(QD^z(T8~5<3^KAptLFruRxzfsXh|w-m%RP0mYrMp~$E_WH'
    'S2NUcZy*yDCc|nnA46G~DKO+62hf@l@%(fpP7oYVQUHDE{6U$Zn^$KQO<^bBv@PiQ@Q7aDkB0V%wScY+Ft;}NoF#i>RYQZcSC}aOidW<pG!b582FbLXR@xuR'
    'w-$52GpTi-IyR|8T{N_-5Ro9+rRYO~?sXvgGE;99mVyyQtqmNdzP*1(Z1fevF+!$OFB8W0pCe)5J=|87)R*cBY~4-*ck!6fIlZAIsX*E)D+-~ctyGZCGTc#s'
    'QOvt?5x6kS*TYWWWw%#AVHEa1wM2#kcOYUGvX>v<2u7J<Uie%`D}4R@E0;g8SwB%2*zM{S{}U_xd5kT8GW4SeCS(eBkG@+Tucj_6sh7jRBR0iLBgeH~62rnM'
    'CFm;%G#b%W6Ml>PNj@JhV(Uqm0Vc8yc^W!ByF8d?GVoQ&@(H=Ew}XE{Zg&_61Ug7M4wHd$n<EGl4eOW4Wye5sV(kQRipJ1zg{>x9`OOkWQuBozmvL8V+t|;+'
    'P3s5$@DIVz#LMAbK`PAp42Y9((4jYdz3qwA%5WTS?MjK{BjIJ}St(vaHv0NFYXM;8N<!gqX{WWvFT7Tys|YFYbIdV4)*g~Xf~W&n<guhGSlN9s9>6atgvhho'
    'rE23vHh73c$Im=U&lxjZmw7S}Z_4jpKjkMOA8L|iiLnYfG(!-0*EFhf2gumLEp-$vR~P#}lhpKpGH3Yq?K1$R!$f0HRIx6GWrXwsi_%kJ`1>eFjnEjwQ(@>E'
    'hV3hPkjQy2;!%QIzLB#aoC2Lf-+k{1apVEj;n^ssa=!is%xVf9>u0XLO26)LQ16pwJs_f0IWfTMqm_nZfLTdc%T<o=HFU?G!QhHR7}rVp3FqRVvspw<G2d(L'
    '+?IR9)J{oL@iR%{hwmsOgF%*)?Zm6GSXmrk`8dw{I<AE3bwI(0rZTVjse^k0UmpFh{DbSxFS)tfg5~#U7yB6*D{|Z7aNKPoC{-0o&+KOl)+G$XfO01@$EwNG'
    'M??aimGW1j`PI~-46H<j<<!fSpxXQQ0pOFQB!&DicRcZIfOO&-+0W?bN@b}C`-|zEXHIKZ1UT-G%+eV;ix<#7U9M@(N|x-lrIF5ydM7_bOmFaKLs-zrXbhR`'
    'FtQ`MJMyMDvMY%Lt_1dyzp|9$Eg{d&0S^ZBv@%2H2Hnt<1&9I2gbY$8$|}nsv2thK4|%?PD{v9yWz7aOfbcbuND^q{gTjtc6zK}0shVKL=3qDX$!IfaT{6Ez'
    '+n{F)*wQ`9yHyEUS7(hDTiCsIXA#q}>|{m=rqw*|5-EF<*K9L_Ue?Y*md16Sba%k`AMKwMgXyE7ufT#G1{3`Q8SNOikhxz92fhz{gS2tls++dX<I4i;2N0AW'
    '_O@3-ZcMOz5f4gMMIc>lsiWZ54;25L1LsD@3TFJ(RlooO!v5DB*wn+p)XCD;)b4LMFr~2>v(Aq4^I5-dKOxb1QF;dbH?Y7#Jr1d0H4dredo3H7R=(|)6tR%f'
    ')+WSz&-P^OLdfOR<CYykXFexOX3VyaLgxK>LAj~ABzb!}bd9zulVR6d*f>w~d~%oP_tPT<IYDG&c@kdd6!%moTDkIOHrI~1Q%izi4o2qv(4Vv9q{Hgxt|Zej'
    '73;=F2f>oA8!30Yy3lKCcbOQ-U4*E$;-b0J(``BTQ_YSC6CwT0;-vGI;WMdr!|jQs3l)+ec5x5XY?W82389c``zewi(XM6&OU)pGLnRp*3^uX{SeRgY<ID_|'
    'F6~_7v<#88C$Z}(8AFdF864N+E!Q;yI=&Fek<tYfp!&>P0K-}3A~d}{SSQ_8i{s&><)gz^r+l`kUzgxKulo|SaFs8an73SYhY|A2_FF$Yv&-ktwAcX}und_G'
    'FQrlHRd32`5mwFE1)D7|8<NLGizBeuO!QW>$=avjRLqrn@HCcyP4BWNNd2XHl$chC^m<i1zWfII)rl9imoo)!1(vk727g(9<m#+MHma4D!>(rcxj9AEYuD?M'
    'lKQyKErMQ_2r)YMNa0h2y~P*-)6gg%F^Xj?#NfalV$29Y{QOndU5G!p5mjTH?(k*uacDLuP<>oQABarW*^IlpjH^t*bG>-ux>hAISubE=b>e5|TR@k=3xj?='
    '-SMl{QBbbpEDlu0BtTDLSZcJI{*vH8#A7`Ex7>4cd{N|SmRMV98z`tf^x8H>uWIadGb3g)1_HC-l4dd`6c<pr(e{b$us)$?wmks%O)|f;2I9h!up<VAP_omL'
    'NXN2K8PVaTOahVw{x$F6uFpKTv>G_jdh31DUBFp}Z60bS``hq4*JHq54+*a`Qss5Q=dMF2C!tK<CAKA|K$~W>877DqJu07j^sE{zH+9Rr#is^}M-9Ix2-W>p'
    'r~7hE&5PMs`MX!tk#6sq^Upuab8C+WofYTXey@)MA!H&Q%lbA9_x+v~GQZM&N(*mkJ6?TktC}aus_jMRN8gM!X5IYyTJ>EhN(5iWNr;7kUZ{r=XhE$fdLiux'
    '29!k=vA98i{l>27_B;X$RdA>Xf?Min(JytYY|yPqvD&}}67P$;VcCC7RO=}<Evc<Y#HLbn-0=APiA>^otR<h#vLJis*$0=%xwzqD@Ub?44ymFTf!xJ5f<06+'
    '>);j$5oV+$!GS=#^I3ts`qgXbJtl7LULIs116GcA7mIFEmJymo!5;glThQJCE#~5N;~D15z>Qyy??D5V8bZ2&YNi$oFp(W4J)pC)u%Rc}A|nM`0PgW?GRdv$'
    'X5g8v+72Hc!`%!UdsM&mNiL%BQl`hM;{Fz}kjHCr09mtAS%GsheDIG~!SBsk;}>6Q4L5250=ER_mBPp&ifR9R&!|jz-Z5wk&Tq#ofEQ(SMO81q>j}AmJR%cS'
    'W@iJLtSZ^`8OQbU@ec?0ImS`j?x3}NS}s6CrF@!8xWsY^RHk599ZWw+T^ToIJbjwOj-ZdZ&eu1<^@g-Wn4jZgIRif9JV>u~h%X4JNxDSWIugP<vr_;9rj+mv'
    '3DQlib(lHFMO|E)bqG=F*(5)vgDuyBm~x-7F5P09A~Y5ZzY#>2+=~p<MPudNj(n5nDXLZDRnk6(bHURhj|TY;l3=(Hl~myVD2TJp1w#CK5V^3Owt_4xq&}6?'
    'c2&qnf*20u?r_jK;NhW9{y5q*)N`AO&8cIzL1&XuS_D3t9e=oy2DQET6#%lnYv6*^8Wz$%HVRVUpKt+7DPc>tuO0wdOJOZRC``3Y0S13JAO?)RpeU>xFfoXH'
    '1;e8{+epxWNCj|KAy0KMLeUV_t8h79>|XYLAoz?Hd&Sic08zBab@&DTis^lZwfDzV5M8hzKdCdu94*OTGw?g%xWN$C%ObN@-6q^JQ$$&vk6<*Le3ko;anvyO'
    'o{`2pu!gT1#y$58#wkG5%wq_3L#MGMSpF7Q`{$d;a>2o2?_8aL;nHaT1IDqHY6PpQw?gJ|*akegO8-pOAGddNxSb>9aR6jA)FEg@`u)iPugobUZ#CoOzFXRp'
    'Ik)o}6t62pVUTi8Ue%ixTw!bKe9Funv(xG?;Wj1Q-&TXozyW>EEg3)hyjLY#5e-4AS8OBuZk)<LS%86QdKT5?$@CD&!5!dqA3n&%z^@8=vGax3A_F4&pu63u'
    'P#?TAgqxr-35@q3Du@lCv<oA$ywGC+7{bk1Fy210v#XQAq?6gJc7MK_v&Bn-MFR|*=J*W03ul6vA90$;HA=QVydKlR(Zy*U?oncS*qmV|;sko%4QY3c@lrES'
    '{vffaJ5U<+wMesr(bX0fhF9y8pHm>M9k5VjLSq^WaCkt_fehvj<8mKYrhNW({S=AiylWZfx_EcBFZ+~UZgJYHkoL$jI2=r8EfAEZ9vIp-bxG*dxTWV{PTk<a'
    '!qF_%H|8PH6oW!<R0Wa*zE?RP_QHXnZJibz8tza2eU7Ifa-#*UY?OrZE13UH2Bgcp<O%%G^uBu-2!!~XD2Q$zqYK-H@Sh|F)_oLMGb~f_BTr0#!?j>kJ(}QC'
    'tO*LmhQkcR2dnsX(0s`=P<wtP#DNDBfvohKA0EQtNg95I8gYU2MJkx-3xAVTb}Qd@2C7IlKXml+2xi-9N+PTj@hqkw9PaBIRj8j7WuE759j+X-`+Qj?Qt7Q&'
    'yO$gwlvq<66nHZbsh&P#M%PLAtEvX+6^=yzTb}VQvHZe-skwj_JR9Z=)4LKgwqh0f)=~plSe~mczOAHSw=wvGCOKj@i58wCA&jAr&hM{@Lyq(<6tvzj&q4Lu'
    '+!V!nkJkr`+GYoezZB8o4{GpnqlLoUZqGwJ-cP(HY#3tS(E>LV#qU9L<6VQ<J%N8tX%;@@$m<Y+fOa^5fH40xrEzgGw6wFdGyfayX`kC~u%mom^#qpeDcDlm'
    '@1v@3*p`C?>_5O2TJE%d8es;~I=Z_qol}m-&_Dg`Nh6|)^Qz2h9C{ULh#%c`$nL?a&pystRITUhE?Z-c8Y20@TXx0y5jv5$Ipmo<%v>?PAAHOk$7_nK*ZdG&'
    'MbjrYz1`KO;g#*)h_2d~Z?QS9c0*~Tsi-atl2sqi+Rep1cl|Z5QuL}j=w$7;>`5D7k!P`ZLpE&(mlwl16PcI<X#O*xyJW(*SWjoQA~G<(*7@e~^#wrI`|H_8'
    'N4}tCAGE}oEt%j*$pLxVUCt?(f41lM<qir7zGvN|L}{BEj}i$GSkN&6XDG_s*U!w#{>dA$m6g-=6Sl!v6HOC^OgnpHNmXt93T>RwKvE~6RkRH{dZn+Y(5}Ec'
    '+qStL8uJc8$CQfU2R@?#m|9G;CTP6}+zR)E=Yq|?`uEEhRej(-?hvqbxf<yPG2Hu~r5Ht8QX9?yv^}#4!%y!xNqjO@#baF8jv1#oO($(9tV3}^dwRKVUq4Sb'
    'uh;V-O&7<f_1BVneSTa#yds~eL=D>Am*z|_ay|c<yUz9)R_`%(dJbM6jAMBHN{gS3?Wm4w;)U1^ABRI3_lV{gpW~!s<vQj#==FYWhr!a@$^@PMmrt<RX;qH~'
    'k6GwD#3(wN87&4sVI{09xx)r-f*ZDb4PVlbl5V=@?Zk@1m_#B?;U}a}sXQiR|L7-J3Eo}sCuozmWCq`PI|%bFE9<zS3Lja5<8BYiCgYA&ZK(C*TosfmWrN-m'
    '$>av$?-Nd7naV;DLxg~W0unE|=NRRu69Z+-wnOI}&Sb5Gv;&y!n@a1FD*8sOVT_Udi{Va?LUJq!X)~M#;H(9CG9PBv;T=kCH9H7TYDP)Hq<XAOH^(WbL=9wT'
    'o?w7kVsP3j$;z`V=0wALh<#<hlcP{=i8aTq^@=kKi#}=%Z?yt1frRKqwF_XCki^&2{OeK;wwZ0Wr9bvoWpuG7delJN2gdxNMt;rGCLGxbiXd``+Vy(j5p>+$'
    'BpqIDH7PPgfeZPRjt5zBvCTD&*3$*QS}|*$dX!ySphiOSkUaUrS%5bO%Zqn>1W;5R_|dW>eb}wh$`^EoE7QM8+y~)O8|2fGpr{%sk&D)Asi3Bh0SmCWV7;MP'
    'bd36iFoHGXvDIcYPPK6o8wJ^j!7<zx#RHj2c_br~json!W9G%-=Mb$n)Gut829bXCBFwV`OD-A2j&m^4an$rt_C17Gf)$xVw=Ne4Cf3qnH%B+%(os&w%N)g&'
    'ZZ+GiVRH=47Y&WbW^M!;2GKQHc7hQqG`=cvmFxU`hKURj{+Nuc&&0_P<^+<5e<Xc`n#EJo>?$+>@URaMLJ4P9@`j^X!7wGVDf&QW2ywkcfLbORl|a6}0@-34'
    '`_4rd@1x&v1G>EWw$w-xc_K-$W3ko(S$g#!EZ}Sy!Nm(iT$vV%7T?oYM4*A9)+GiH5zXivE~$G!AwYZW*jj~z<uKxbK=z#!RgYr0#oS@}Sau1J9#Iim%gQ(0'
    '4+PT-IjnfQ-6`nWC--C;jSz<n89yAD<=pm>KvS4bh-uQUiD*HuL6}l;JDk`-&xK$jicq-|J>o{X6~Vd#Aotm07ad@)G;oWQEB7v8wsmU6_ppbin%tKPp78#x'
    'HLz;e5c!<mu*g`kx0wx!F&Y@=1tJb7g3W0S;IMY#i|JDhsP`+@@7H<}H}K#mA(^+8UPaov-$r$otQP9k{IRonE-pp#YXN_c>ryc&r8DIBTKGev{iFtWS6Xwy'
    '9Ztw@gc(6xt3O?idtrDM#p6L~8Y2als=D+fNT_9Q;c^2`jc`043KoooS)wt&Px+LaSISwJ7}FF^!m2TZ{BXv^#zxc+Ex3@*&x4w@TqVzhNK0SO|HiGs*N52l'
    'rDr@#S|Rz?^5Mu<Yv9pD7!yEn!YEf~GTn0*Sq$9&`^WH}M@@LJ{!vG-r2kAF8KPPR18^*Ql)#g&NU8&+EktJ=xxfT8gUZ4Z@m#HwSp1qcKZnlfJbBe^la~Vq'
    'E(uvfGWO@CJi$*k*zVv1IHdZT2Wfvum>_mXfgw|Xh=o4QpK;7&?tpoxA3{^aT;jb%YeDJ|#PsO$xAsX;{;(01zz<!go@x=CwE2c_1L4s~n6x3H4u}9m67~;X'
    'LxI+(ykWh4>m*dD6F-OoR$z4yja5PV`BwZYgmN6+rd@WsHPlw;)oIUD`2x#@H`JWk7@ApueBxgfxDn%QN@KfY7#84Pl8ln~IBs{E?XwT>Yi!3NU4w@OF?Yk_'
    'hz8|sp+K<@*hmmJN}Kutw>k_e8vY}z@NjsObv)oX_vMx|;|mFuXFq)UH|)E2?C<%H>jZ5@17%L7-Qe4kYEjeGJS*9UVd%w@S(D+~tw_PJcx7nPjEqx9hTy?|'
    'A`OJc*?0vT4mv&uH#>Koq!6lG?{z3nW>mK}ngQry&vO4nB0D{)fcQg(>EwnzGT_AsJ2T@n$U31yhTO}>6<M<_+RR>}ZJhW!G58ArGELZpbSCat8s&$|ga4Xo'
    'sn}sus4_Txz;6m2#pZ#oLet4zRfW~T%5^NltKQ4$mzZuQV?O<s0zr><WIqs%E!hZ4yDE~Ma8s=S3c{kYA*Ha+0H8cT%&>TY-wR5x05BS!Ox?u+702o6pS{>n'
    '4RD`PmA-sIljkjJjrJ*xhE^)<&7SOdDtKD%32t;C?oAY<>N&#Q2Uz-7(9|GM`b3~DQ!ar5dpX!M4duGEtHAG;DSs?<P><s&HDyRxFvGl}F7Y}xoejrDJ&`ik'
    'tb*<m@i&4b*&mfpvW=3BY-3+*?_!Qcp`8#srPQ<mY}0rRn8vJ<&F2`~-VxA|5}9!A?Cmi}bUJ=j=r0>!4;0?1O)~(#X5a4LK(R?uSh_Hwu42D!xmyH6MO_-8'
    'A{Q+2Uv&*p!^-;yfGZSWR%B@tY!?KXQ0GFvUJcX<Y#rzXS2dD*bh>eq+SEAu@x5ay+2i`FvrrZ4-1Wn>SAKihc5wnqgEw?;KvW3il2L(Vw4kw-r-DQ}D;yy)'
    'ltqYv)N$J2t2E!xNZZf4MQuxCjwD+qSStl}6_aK@lT$4gKu=VxmDzF<?-UmQ5JlBMKGI$O0SqTaaA%~`>#OKFSZLp3#}qQQVZkVXfmUcBT6I@1M4W#{noPBp'
    'If)XamGRTL<^Zj85vZCLo)Lr33fhIv;&#gfJen-|R-^)Fcmf@PB#&@BFd*5T@g@Xqb*d(l2GT&?h_>E)SFPcQux|Zi!k45VXFwbd-hS&uJa8$MlSD;#hY6&7'
    '3dMAdVEnDgXR>`(@@whZUkWcA=IsGO2ZCp0x5(l(SJ5)D5(5+W<-~xsCtggQgA{sSz~m0O)pc!^|C7UDyDQKVL?Nj^3td?YnlFq7u@0pbQ>9QW(cE7M8wWpf'
    '=$*wjNPtwOVUM-AnCbgH2kYRl)k5JXg+QJA_0QK8+QzRapVRH4N}mo`o^5cIfmtY8iz7l0?A;x9NS-JJgUNt0?q8ZRy}%zC{BBL5ZCY|hgRZrFq*52`Uk0Wy'
    '8tv>%*2V(@@gG8ONTTJaR(Uhlhi@-ATm3~bv*F49lnX6E9W2cb#8Gawmia9GrZiHrw~`lUxe&16Q9?0P0?O~HGJ+6P;c7-dwqX*`$Eoagd%6Q%;5Dst1*KIs'
    '5qM$hgra4g8hDKF)wrQt3TO?UfO8L3M4Tp?<>C^_Gy_|A!JL{S+~}?x^d6kk6eSvFz;POrY7x<$8!T%Q{a8T<CPBcXnj<w)+L2MXN}6_6nsKsD{>)y;8VkBU'
    '*?2_9gKigaN_xPHo8qzXI+ptXyf?2kmVj#{S_u}y&(}K_9&s50tq6rhB!I>~7cF*u7`Da{)<sj$-LBWkHmc>PDJnf1dA^y)<xA5>DNl%)a<p<)!ZLM*KQR?V'
    'n-?}kTa?3dSHCuj^1y%qf@9770XC^MHx&XgA9O9YWn^Dxf-C;02>P@q_EXiwLxco3D_@S%_l{kJ^JmhYpm0{F5K~j}RekW=w*O*`U=M2vcwQS!lgsvg*}6`B'
    'w#IKS9n3dksVEPV^<QpZS=j!DW?Aa4KY&4>>i}6;52kCL?DBs=AUSuxVOBUVD|&f^Sj!ZIk!S>8-I4MYIoS`xuID#^Ek?Bm%T%X+x5pkKrN2bSJNUa{oEM{0'
    '3hDZP`KgcO@zm@vWm7B8FShF+;Row>b!JD+FXb$cK3Hl|EW`Rbj=4~ZxVhSH51;%A8tZXMAUC0aoL2tAggFbG7lQGH$|#-#ZwgW==CPC;W=q-3^!#iCx7eUs'
    'NQ92Qijd7De<T*z=<$2%O07#b<9^yasae5+1sAf&>+TE?@F;`YnuQjP3`iD{>QL2S8E`{M6bWqc@X8oF)M=Qj#NjR>RgOyQfx2)bykG$7;C{X8bX7CD*G!tW'
    'TJ(?7nxs)WSv=vxj9+#xl257Py9|M!JO-KNV(>k4OR)6jYMFhjz?$yrDVDn?T8>ild+W#IqjUKvJNr{z;yto?iC)i`{ySB;B?lOPRw((9C_olXzTOMn8BE%j'
    'G~WcnR&;R*UesVTsNabjoVYgojNZ{zaQhU>)nOdu(=1A_agDMVdQG+nnX4qR@p?4R<|FzlX&Db$^ZKC=;yvz_1>I^Y1CPm*H(z8$4l)RBevjZdAj#KO^AhV?'
    '_hhGSj+`kAh(gEqc~7imz`t$c`uWFLeq3j>L2>l(MC&k4(w7J5RURR!&TJM%JgM3+XQauIbBiyWr7R2aa!v+jP@%|5C%0$^Pn%X<-1w!l+;M~68{=X&k+s>#'
    '+}3#bQ&k-U+ixtr;@#fyI|EptTp?NVuOwrrrI$9<5Hqgsf!zEq*91dT8>?iC7&^5l*Y9eA&X&2B_U-YdG=B~FG2Ck<;f>LXsg?t0Y5-^&%1iAvlyL+ygVQNt'
    '&qt35_(uhJb)DL`HuQQNTcw^~%2m(iY6|rYy!TJ!E2lawzn;|w>BcVy^UNLV#>JW%^$GR!&+R%A)GIG%7k)2|%El=*^HE3l&GwaDLi7Z^^!3VO+V9L?Y7thT'
    'VJ#IK?)c@+V(zzRn6=7heD<!c_EPrC8PNWC$;w=y>S;DGp0S{}M`|47TBoaZuH@hs61HK$%=r@*KgsF)i;ATY#2l+vUVd~q%ex$*5=>u=O>Hyfz==-V{PC5;'
    '`P26K1x{DC`+czw2JF3)@+#|YOPSjE64dN7b0@GpF8jdEW&?Cki&nQBql>>rBUU?6&d(3+E`<pRy3k4T2~IZr{V$tOeM(%U&;Ba&nK}Qu`k%`D|J*SA@9O=%'
    '|El*}|Fz!7{QC=xL;&BC+}hqpo~Go11LyQMoaXesTn2Rvnf&VQ>3hquOgJY^o58tMZ3J!GYwzJM7Ei=baKrjC?dsh@ado}YUj<N0XXL~$21Eq<=1f)MbraJ}'
    'cUCZl9f^UZ{I^MI;<4)nakInW{LKWLWXSAeyxyBYvDx;dsvX6_`uyr&CXSkF%*mRn?2wRen?KXbS$iTh?53`<o53c~!qwpQ?=IGm-a<jA0GeSY=e!$cvP;Tz'
    '>-l~Lj{jQ*yxel37!qvoA}FIwZ9)@J3*<5E8Ti332JCx8N6hU7e7OVuE(0w(yrXo0-n`yxYx?@YAPqLW?~j9*hkeP1uZM#Ryvb-&RTIet*b498%upIA%cS&*'
    'c7dSH&kSYjC(-5^RW=~OouKNP<3e%3Cy?qf&ACNVykIt2hrCwJx92}UZcsJ*9HaOA*;Hvz?2v<f&Cf$9(NWkldtn?K4r{-%MoDAh%S)Z3+jTA40JSmk6O=Jd'
    'ja^VS^3q3_ueUw&406#7emnwUf8_lHK3(3<LxO^W5^K`pW5~$1(KV*0U}N>N)!pNcSXPG*gF~+OJ9i_{Y!g=qrK157X0H+VEhPkUf8LWM3su|xu+uh1dbaJ('
    'Jug5q7{qjdM@=qy)xK5hlA-9NW&MW>%>La4_B#(LgjM>VqrGD;KIM>&H3#ZZOkwSuv+g+xNfdxPCJvG%Z!~eouY2Dgb>tma6`|5f*c?6ZI5eFwTb#6^?OS8='
    'r92TPLl^@fydr~W%@f&(LNR#}X^9lYF8~9)e9AEFMSdhw17#d#;I`o7e-`tH)#xD!Y6uaE;D~1vfiL%Oe!%}SRklktgtrJu!1w9^Mrfk5)(87UOZx?Nsa{p7'
    'ja8HeRSCVON91(mNRrYd+Vn#<9f}x09oL=rb3#0vJ&yK}jX_y%P6=VP)`PCk&7Tk>ziJjE1v4tH8O@TGd_}tW(mPC_H=|72_*GqM$4&<eY1cK$Cc|`Y1Hs~C'
    'H+Ymyqh{YY1;4v~W3z+#tf-DK6j8NJoo+<D-OXy<sIz9pHq)HSG;<*M*g%m{3R2EiqM(#_1*;CyM3API&4+pI@R`?=?3t)l2({C$fj8=oTCVMcU<%Qx)IY2_'
    '5}<-PW&j)81UlJ|S}+FO!}t#<;1|IJS&znEkzG5|!9}VQS|bilc~O$|S0d+~fMPZ!Z3!H)EPyzLWV^z7qPI4J5;h32%<w0*W&|8&q~>IB7$h2aiK!6_1A4pI'
    'fhZ4}fj*GTxwI6r$<ihX6uN9z8`<o_OQa)W&N<tF2OzYq<~^W8b1m+RoK)JrzSuAkf4`(*nR0Qk|8@Y%#y*ogL(b=^toIV@i*>UJ7DD>Psp*MIvw&(uX`TOq'
    '&K~4+gW-KgH6(|K@`%62I{a3MCp$v7;~E?E7}!-MPU(&*!Ai)|2yVpr`0ttE<Ug1I90C-`@t>Ih@gGcZjr=zzsQ$k&0r`J00sp^e0%fr7pnqWklx>}=e=tEl'
    'd+67DXoDN=)RHxnKKBP)Lb_}<mLoEphDPa55EEHI<I)X;Y)!~~%J@x^L*fNzW%lSrvIN<nCc<J^-6ef@+%s#&!ujHA)a4c`+TKmGfk2%)3|tf0q(qF@!)S6B'
    'S`hCUO-D3wXcXpfKGel>gA|l^s@FaOzSL@g7N}WL6CYCO5*9X|Pr%&3D=Tq;A;d7%(gSBWp9Lkg9POb<)gBbQnj=`3myE1zr~?R6Q}Hx{*rDsW@+4?PL}g4j'
    'f|~q|NOjM=2J4tr{1^OB4R~5iJ-Sij9Z2p-WwzCO2EGDIe}~;)b(~uj`Mb-n^bRP5E*4`!i)Ye6g<|_LzRO7DxKbIw_Jw29PW{ekO--S!l{#^SZpqW|7<Y}5'
    ');?AE8{i{=vC1Oxo{!gkd$T@{*i2yIIcz9Y4d@#|!x{3cl2$lNs7vc1O{@VW3I{Da;!;i05hzYk@0)SK!J$2>uPPK^2g5iv_$MOn>_uXP2#eIO+I$i)hzTO('
    '4bZuPP<Vlum6&SZ(mM4QEi!PBcKu#T+1?}|HTKvE)@q50pfhoG+t#RacJM2mCuhC4c?;|&{vnEseWiz(Qm9?(k$k7$Do!4E;93DcQDx>nqVas=>{vg$-KJg@'
    '7#v@B3OeidK~t^ZiGfJFAbtUNs<@{Z@T$Y2q3<}lfr>ynU&!?xcUoaLGCz`5yg}tKK4&nztana((kkG@5h}l>>jv48Jx)qdaxV;P^c9W95626sNBtrUdbcD~'
    'wt>5EZ4KT*G}2&-mz`#e<h<CLajQoEL^-gM_2CKkoJUJpf-3`8!dpmN!F=$h2JQ_KrJosmONo}q<4TKKCFcqc8h9-pooh<BYCd^_v2zh@!{{X-m1E8ztTufG'
    'Un?<E9nev6m%$rCg(;5O0rD-?ooDH5&o!UeQd3eNC|kkGyXc~sa)aZkH{suH)bI6bi4O~@YfMM9DDa{h2`CE&BNJV432kQ#0YeIZu&`l08>umT`Jph|hdLpb'
    '?UayhM~4s=s@fYOgGLJ@>ULcfShOC=7Lk!Z`*}{4HrJcqP-zUDJ5NV+L=BH3*RvDvb%xNY0OFBz`0=Zoi>_2e_V+W8-pzM27UO^s&~@jk)tNy6A#GzL2W4|`'
    'gm(3sHDyz^^TP82;JcvSGH|gwj6JNGd{y7Ohq@Xyj>7;_^*34VQJXQkaU}T|{`-*omJ!H4GDT2z)K!5eY>Ivb20sVz=NP^j9jj7|ZO%1kb~vS}3l-rK8d~$|'
    '#up*V>=ZGhQi<n*`)@_kau|8&N?|I4<vN*WMbIGy0KwaF*QIY`UlWk2vF4Z6tCh-;)+Utqg<k%OC`^Z&lHkymE}5MH##z&&5knrflXD-ef-k405)h*$nS~4+'
    ';?r&Q@TIN{d>D|9?E!nE*{OQ$NqGlKQyzah#U}lHEr_d_s_?V6QNAK#u5y*^qL0$Jh$>GBTQ~kfOCt{nmOA}etW~>@qGw;8?T|HZM*o2sr64j~r4e80Yr6_V'
    'wHr%3+kWvlvaeh5TjPQ+h|RH&N>W^T1ga=#8<Dv43p4mqg64CU5roYVek!a6zW#iVU~AdE41(9anr1vyHDT|M-9Oh&R`zhS9^XSAq*<w*YdAQ5*Z2If^LgB4'
    'iW(bKQ28q)MhkTPsAi9`z7;trT0NlDu9Va#^EeJz7q}r9`G-N`MT+%$1`ay~L9-uH=(?M+q>(>!#vpf=EY05^w*7s-v5TPvRbkw<WGu-ZNa^_orFM)I;?Nbd'
    'fgvnyJQ&fh85-gJaulY*=Tp;%KM4E`V^j9?g+B2<C%1Aw-|93iK5<^>dZaa9-7ws`VM~JxkTqr|g>G2;`)m>1u<?3R6O>({8dCj1UpRcftdO0XG6zG?4ZHz~'
    'E4Ckf<EXVRwnj_C!9Ph~yB>%_Rj3w(^OnaScLlnBOA{2J(E%%#+C#dV+w6*BUmDDdn*W+mOHAKOSXmT;M~A@<!dLbyequ`Vhf_kW8YJ&VBd1GL-st`53x0}S'
    'za$utT-uK1foc?ln{i;~K73T=f^;IT*?R#hI8hdSkZhYpOeI|p<~RU-V1sd~wXxfIa?D&1ZI%biZb4&6Kzn31uSfN10TG-A4ufrr)j;Y(Lh2-EJy2%D!9AN*'
    'u$nRKe|E8YgG&b4E#Vk{gB>}}ZRWA30Q`M4Z7>pzX(3VzAH*x#we&msG#pY58RJR-g>@-R=KLh;nd4VGb#8mNX&djbfv=L3;v%B?{|yT4jsHCqnE!u4fsy<_'
    'LjmGHpy0><0~EObGZgIoA5hTp@1P*<|AYemiv1ah|AYdIzd=EFd-{JufrR>hK!K;}e}e)E@Be@Tz5jrMu75y5%htbvg6-}94h8lr+W#I36#rkKfZxly6CdaF'
    'PO}QTRx!d;45kaW3sU6HtNP)zzNuuhHsDo<ko&kue$$TGmf@IP>nGcO+w=0mDc?c+?6|x>XF<vDXaUvD)kl}>){aeKbn%80rFts?P~Q9?+6fVm4C-u?N7^H('
    '<Trrm82wskc@<7*Z!F~RLl+oP1HMJwy)*yrdE~R5MDVvM>@JHYMVTLs8B!!pa#WVvaU!cPc$?BSJjC@|r$O*fKkm(G7t%R+jP5){Luw`P;M8Oyer7_dJ$;%s'
    'nRmM91szLe47z}1Iy8-7Ftp=AOzQfzWB|~k4w}GOTv_#o*|tg_MsOSu!HIUmuXy7~>zIcPNm+6_E{y@!>Bwzg8-tjnc>PC<{A-(Ii{Vi!b~(=6IgfMiWlwOL'
    'EIi3>YHE%4QNC4FlQ+!pkZQ@0+CV_?R?N3GxFWT~(^cH86SsH}XqQzPa5Tos4WiyQb@kVu?w>nZN!^Q|YR!MZ2CV=`9w{h$;(E4v@zZ65@Fr3_1KlK%bmE<('
    'ML{2T-|=WSHR!FwYdm>`AAqG=uSef@?^g@`@?MUM3tpeeA*<H`vl054w*iTLu<iURKkp{3ZoTWTXuQ0eIu42Bf{-DpbsN`UT=(Uw2s{KMrqiOfF{|;rX-SKw'
    'QubJnj`7b~k;Uc{dObW)H>s}Us_HA*(@QTF;MxHvYV#fH1X&{ay>C~dcf}>C>F6pJDgzEHX}DNqUGaAoM`U_7T_?(L=b_=8l`QXkmXa3_GkLqO=vNa$FEi&W'
    '={1Kxm(Mj8lJMU(1Qbv!pgn)=+pR_V3Wu(n4!e@S57q4Qm+K5c2}7zvv-~dZX5^>>byXgB6<<n!)o}<;@yzKf7=0lAGZY*T^VElc0RgcP00ANWHz=^Twf+0~'
    ')JygC=yi4^-)D8C4k*H+>LrXy45-`?*;qpGHL+RJX%O{%TVm#-_&F`vnV$@0unZ_<6K<Jq>t{kJtOIGgTr2~~TSd@Qg2^)u=g7=yI{NzZ)%u-4jFUG%CLM%('
    'xhe8I42v94hGUs-3}aT8ESEfcNEPu(o)^JxXoRh6<xcSDMhz+T<b@Qa2gMYVl|f)`l-eJ$F&LjPw=Pg{xrR^;;)fYUza-L0S{y&VjIyc1?~|1M33}%;nmU7U'
    'YVXXbgSkgp@;TW+41I#?1(6R*m)GF+&OcuHTTzm0-i%-1KR?EpcG0aUo3*v@7@M#yBcQ1}nyG`|Q*6DLx&#_)|5OE>;24bz^<1)1vJ`4g#ukGziR=)K`K=W('
    'tr5lNozN9QyH;N%1=aini8leF$)brdUqmqD-mRuy#WN#XNeZ(_JnO(T!y-0MihTYxyG7;U>L>JPGAoeaxSm*`^HbD;-GCp8Q5`C9yt|RE#sL<37fDm};u$xO'
    'fl}k1j$$FF2EqlE+ArxEU03B7CyOc$LX&O-uh@bkA66@mC6BccL^mA4#zx~_R;j7av0geU({hU7rUc`mWA1y&RE<t!XeR0O94{=`_}2!TdI+><I;{Ji=A>u%'
    'IH6A%;ST!3v0JV3O(@ZPycV{)KNLpQ*Btl?%33}}qaB=h=*X|;li;P@`K-uMtG;xTL8^`F+>NjSS`>Jyiq65kUMZ34EU8MWGLF6I<y7<Cm3k2D6J^0WU4?XE'
    '8xuW_x^w6^@e491Zqvg1ijGT4Y!z!JS2h8X@uaiK>+q%n%s%_PguHtiuqVAGGm$o`JSn2zJZkN8AAsCn(dvSBsSu(MC#^<BaJ}m!ovUWbwE?Lw&i2KR7bv%w'
    'UjdXW%BfCX<$vgKwadq7v#AWQhT?!KK~C@vj%D!l-%T1ExxyuaL<YL@*aZ#nOdlD=s3M7F*uN$}bgslVbijw|HTXp+lO;r$2*AaVF0-zV<Tq|YH~m7g>@4C-'
    'e6v}V&=6XNE(H)n)(adpGQ)#r#qjq@KWyCM-WYf1mW4frL)ljjg?>^s&elcQqno&R(8fcJEZ&%LQDMK>5QArnrLv&%#^1%(0eA)ukw>gd(+Kz$rPWKn)qrf<'
    'AybpSfe@iB5%}A#>dhY+3%uG9<9T8~_N-uXQ!FK2?&nm?POI_xf<zYi$%w79lGV+cV1&k!z@FG9`!7KS7P#7terb}7cEB(*)oZKEXmw&1K7Y^VuwBF1s?vLr'
    '`bxj;F|?CZ*$Gc6yy5;xgdfMcA`Q_B<36IR4;Uw){{(H?*8X@DR}ErYj;86VbLC(qLR{3G4rSBethdtIw$3b<kS-1|DXigd)ZqpH(q(0k&uxRBAS>;px%w)('
    '<D|Zp%dMCFE-GhUQ@7a~sbd*L$0UctH38QMJ)J$5K0Z<+$<_7a-8LTWR+w5l&guL14=R|2mNfiQ<2%lLSzqb-G`k?!bLZswW1xQxwVm^3uG(_x`_oyjlI)d$'
    '_%P8)OD8q4=U;TwFV4;r8UhFi3=Ief<-ZoKznw~SHg$3S+hJ`DZToe0RKMpM!p0CO{}Rs43sMNM3nCiET*~0)b7=!Wqv(*1=`2}Uq4?}~_do=tB1Q|S7HF8*'
    '+0DjFR>tY=V5_LiAO$TVtW`3~eY~2<?o#A&2n97|#Xg8MZ76EW)(}|*Og}?BYrL9KoyGwgI!ISSD?Jv&)oKA0#i_u_W0$7*I(VQw64=}N74ZP<3!x_=zC7w)'
    '4CbfnV9Wu0oE3#|a?=qKej%@IO?9P;Va#x$MYV&_A=b8C_1j2Iz6Qa<_ZC5EZU64>^}V-guj({hVgm_h>J`l%9tq`s<QN=UVu3M9FPNINjI}M$$TVOxNDxak'
    'UknZ-fk+2~vZ6HBtZSh&8<ACja1y#-K2;5qG%voP!0cgqeHrR{^+#CJB+IyClmi*&+7Ff5<94G~!rw<hIAhVdmeCbwkx%Ae`0<T?qMFjFoyYKFM%N$@0)5!o'
    'LG{ouk6OOntrW6bH`FWnJ3o0kSOhfQaAStF96PtJYbH`Wd|Ubs%$mBXuCKx~_RM@cG^^AYB)NEtH?;Oi6wQy*5~kz#b>fL_Pr-4aK&vvKk`*PsU0(}q-8guS'
    'nW>)MJMO%kJHE0GGN!zmR%<2}H~oCyaWrk0GNj3;@3Q|FU*{N|3)7|PIC)~*)`^W1TPL<{=ft*c+qP}nwr$M&O|PEruV>c$s=exO)w=h!t9I4hpjM^eObuZN'
    'CWR_aT-ehG+`Kit$|@g{=$}X_V=iNy9_+7lgj~j;Mm!}9D$Tb7;^~AshNJSrzRr3aBV0US9kQSbWUppv_i!kf9WFqysiYS}-ZUm6)Xs^7FIuOlQkKS{t@+hH'
    '(w%&5F1WNOx)=ubcTf~5l=NX&=Ur)ClfSTMgr(7`d)2+A5}~E0f7Ihd%FN=ujHpc3x=a)_bitORPlU@G4<2@|Zb^TPo7a_{$3m+xdh}QcDJnsYb1JmJS>SgG'
    '^$;9p&^OckQ}2$@ZZp+C%o*mxpD0MqgY2dk1X{hkH54!hgh;^Um({TGSo+&`FO^@7!l_U;{V_5Q8lxe?`O2SckYJt_s61@QV$<-kEDzW7jjqHe=RiWg`8GT='
    'dz#VMMmJ1yvpG1eq@CnYn5|P3L9W$ebikq{s}w_$k>g>uC{XXERy%*m<^=K;S1|Fw)bkPhaSnW9Q2`W7b%<-G4BCt}kSQn<u^`)Svi80?cseuYIL~d*9c6=N'
    '=!Z!I=TEvP!)ToR$)x`4DE?weALr~8!cMeyg(C)<#S6DTG1lNHtPj#bK2*mO6agC(uw|hfMZN$j6^~~~d@(AlQ%3MO@7`k=Z124O5t5Dudw%i)3v4CXRn;}o'
    'YH__dtt?uSek=r1;DEt!{afK9+=BOUmIjO|vU{a7zIHvq@PLvRQLu9Mg8UhG55@>vK^Gz$Tsb_uC;>wW|C6V$)kcM@y6PFzl9p-vhcF^*TJ_DEv|UOU@>xg>'
    ';DT*6UjAes31`4@d6C*)rzg;6zKeqJTeGr@?hNE)<LT<W#F$*JO2;hvDAhH4%FcR+d+_eXxXx^DV#=T2%O7Je7K%(_hdqKQ67q4N7`Z=K(!t5ANc4qPNd{Tb'
    'FtvEXI|-8X7AezLZ4L5b9$sw`EeSqvIIwvv;ux`PfavVy@17FEeJ<;oh~gi%u6%`j6XY_Nzud!0Gy`)uU+&xN;apq`rs{2mbpf)@I`60(*IH&<%-4!nL-3m%'
    'ooaS5U5gPdRF73lQ}zg7%$s>w*P>gC2(s+UU_mEu-Sb!n8aFV@o0`~L7prBHsiF>;_29>*+;$r|P-ef<-j;w2mg=Z-6({u{)6P(oH#O+2rL0O^e=H`7%iqJS'
    '>4Z`T5Vk7YGScW4PcTlP2t*rt(A(0;g3`DGdkT|EOfIvqh!e88=$NCQG>9<^wz{#)Aj@S-Eo=G^+7!e<QF&6>#!OkYXH4eqLh;hTxV-b|hsrrv({A#w#7|Hl'
    'Ndx<Gg0;RJ9`+|zFm{!{@CHr#p-Oo66C;PKU5jI}uWtMqxq)YSrTOi51q$75a<OQNZqOc9vGY2MJQuc-=<T%BJ(9Z=41Jr=)SF(Ds3T6M-SmdV+_{i2SbYm%'
    'R07I=fd4&ls9=N!v4H~tiNgT_Vf<I(_}_!BorA53xs~xh*`rAHU+>yszpr)s&L#?Fl~Mk#hBSWvDLiAymGSS^=SPIX2y3=Yx^C=HL}owrp}#3rU4aRn^8FP?'
    '?B+RoGU|vbQqryrFWc0vojkKG8Pwm~p<05zyzZ+4<w{eo!cmrlu?0{RB&n3ylWqUC>!mm*$W<F;!w`{gMsy*H11Xzf(4mH-{qW1lp$I)<=xnIuA;u^3Cp$+c'
    ')8}`=Muel9fYW74aRynZn}@x*u9LmrA2O#pbLA_Y-YC1nx$zpP5qiN;FT1(@%bA6i9G#r5X?nVu?VkH;zSs#@;KwkV!dk*{1$TZnNeN^vDr*b*u^Qk~jEMd*'
    'O-3_T4OIo}{KPHH#(FFBrL+nJQ(v^@(#@<!aVjJ$P<g6MUyYsMlta=y+{D)4;nSu{h8pKEWn6E?*m862>%O8}(x84b%Ljvqp_C;Yd39-7j!?>WqlZ7xTU?aQ'
    'g4SjdtwL>UT-5RAN%tTZP!Z1cgji_HSVRGy+F?;s=n*lRU|X)FV3Os?IE4g9+VkzR8jW^sAEz*H4P-Jb{^jqD{hF1#GDR7^BrG>jg3VA%0l%=p_Ik=22w&tP'
    '_*?y<rLHAn6|V6XDXlDP1Cecw{r&(#%2$nHW+-D+swvz6<O%|xo@xCKZ<Z-YKSX+o?=yqyJj|jQ9O<b`RnB$+x1MJS{K4|busHg{QK-L>ME1H4B#i=AhXFJi'
    '^6ZH>Xo_4P;Yt~=96=`!<ucbHz22`3^Gmhj!~Up}EtVz@Ka&?B4s-VisCt@vfLm6a3yD^><A(u;*E9xCnBUZOB*KS>=UhA}rjqVc{@`NG5CK7K#+8)85-wjE'
    'Rm-~jyM>_VR7+g#X_U`o4GKt4N`75I3Rt%1yhnc^gyPVrL;BCtl-f=j`^WnmOmW?t8QZO`K;+s_=*`553O~+I-*aDI-J3Dn2E<YbRCEjM2Bo+at3~#bFrA9m'
    ')1RvmO8zn^9y^)l-*HoWhkA+@pxk0uz;-u29@4{dj@@teOgxB{3T5;Ztg3*0gg?7X4&|a631r54z}r_Z2W_44w1+}UntTgRFE>to9mMCl&$SqBI*Y3-_+d-u'
    'bqgK#g~mI}HjsAOo3IyW1I|DG@^C%i9zO;~MhZXoGZCDA(yyW;IKfOFV8!F+>a+mfne4ydY$)}`30<2H{oWU{;v(fk>_@Ey%2}I!Lz6r{eerYtBqH0qZz>$0'
    'a$+NNMp$0^1)*l=P$mpM5r&8Vkv&XIA45$0z=m{M=2Bmq#Gs{b&3Iw<sOg)*<?-j<Vvm$E#~HU~AMrJ@Z<&ul<IKzo*gW?%!f|R_kX8C5mwg7*o4zeTTWLDh'
    ')o{7%hJe5h9&CB|GVqvgFZ9=6Q%rH&bbiAQC`RvwM+sO|6G?3W;r*jjxGGl=a~kuXRpdB_KDGL|;Ab2+V}KaXv)XTcge$p(CB4y(bhZ_8>2OGVGDrnhF4*yJ'
    'r4*4hU5)bJGkL%YG5G-n5og8@&h?$L$0pCzTGeBjIbPNWb{YNhAU(aEW<0B*Pm-In_7=FCy<|4QbXP()!^uo|ciBX6M|U?W3L`V@qKUXFD>oEcfT%kLWq*+s'
    '1Of=9kat$c4?fn5Q$!V8M}D3<_s$$SR8Wet%Qsh9k|EHN=P<Z!3}sD)0Q_n9W-<F+@!#MF?2uC$ofytKV;b!~{vh<Kx0>EgPKP~vx&j2;oq-46K|ivc>>aaN'
    '_YyFZ9;Xk%42;l9IE$$PiA>dXF?5@?eJw+O|6OX0tG@g-fq{VPL4bg;{!3~djQ^2YBWFW%19K~LC%1pZ);UgJYJdSTxaUO`7dQ`3Gma`h2;j(pWE`&esb(|f'
    '3}=}TUzqUavEX97mH9Qx)^>93)t188JoyI&Qa0I%sE|R0R4=qO_JRm%tc>&}9$g8V@^9~4J`pz)85yy&U3F{%JeFZLrDPGFc{^WJd`167bUpPrUM{e@i>k^Q'
    '=>t!U;`-=tBUx8NTEfeP;~~!~kmyV8pLk+~%>rq02b#wCJjZ>my78(R6J3Ow1>?`J=ev5(oN4@nwGD&s&%xl{{ELZ>jMHRr8J`R4$`N+#t18-o1rRM?K782f'
    'VSxx1lK~LpAUDr>x_K|oGLGQ>!B&yD18NFBGiw5J7I9nYNVjdAB|#c#Hk+s7;v!g8(u7L>E<eB|Bl{ztJMJ;qZ|9{F8lgL+{lgVWm<+Vxnzh`Vk(ha$dGC*~'
    'p*8;JArs?-uJ8<Nk(T1!+A%)BWBLq)z0%MB_~8n%ku19CpKuu$2nglBR^<QuANQ}aN!{z8vI*s9C&%B8QCdTpOd*ryiWxy1#S|{KSR@_})Sq9Ebp4oEKAm#S'
    'CdBJyE6zRT(j6F|M4UFR^W|d;j~PGartQabpq_oj=%|yz8<&hKAs+m2VhH0Nb@;o1R0)i9AIBA|mQ=I4&FC<#$UkTvKZ;ImT(y=>?jY1iicVrYUhSS%w&$Bt'
    'Vr&;QX|n|i!#fpHixz5zR3C?q*|Rkx4ty$=_nl=jWb%L^a)mz!37-Ml3CfB$W+)t!<tw0OoBIZSXFcR5dIvA#`7|B|JN}_!qCrzH-ZvBp1xufXgVr<GPvPAj'
    '8;F?@ZLpiC^kodxg9SfEOp=U%gPHH3E;dk1BGcrMIa5mAuJ6l=g?ihs{aYdDkM2a5AdAkrVB*b5+aF;8Y3i{f@tgWSP<btDE`+i~&uMiFKU{+&D_l8F5k7qD'
    '=*I&ymi7+559i}7AD)i3H0_-@Dn{IqrqQuM;%PANwvrNbP8Y3)RX4^DhadTy0KD32Z(OZxFN=A5ih`JihKS>e7K)OBhK+|9)t82cj~yG+?W@a|ijImI_FA^-'
    '-aQap{BCtlL&QljyisHEhDd-A)Q2MOArbRV43sAb8{|&0WjaK#(cUI;h5<Mq)x^D0Y*g32m9>@CSP$dDY~v^nVk4<ncOi(up=xLjzT55oNg}-x3o4^Y9P3On'
    'rU-~|&<jAue2^&_8Q6z$QGu8$xUt9R+Y#9@6bk1cFMjo*bvNw<jhIAzX2n1prV!rYNRz(k*7Q7+MJb+BI8A>Y4bY$iV&H;Gt|u~k5dD=@5X!<*j`vFS%Uq*s'
    '#cn?xMpUtYMCKw67uRX%Vxi4=rRPe*+gB&VFd&+9ec%YW(PSr8Ockbfpa{;Vi!@q@oq-LV%CI*Gld>B>4Ntzs(<s@0repxsf&$gSL(gbew|&qobjc1g5FtRD'
    'tA$Kgm=Pfo0(@aw#2y)w4L@2MXBWbEsbGFJwab@+&wO7wc4rDg@#F_HG^!B<EkmL?V?sxV?(vdIP%#m^Lefqi-9K;gw^aNs4dB%i-VFAX4ki3dVJVU}Smcf3'
    '*!E1NUi^S(Eo-XDgv+Qhe%E{Wz>#>r5*ehoP%@c794c(1q?lH)Zu3iaAaiP-MK;R5$U}y3+!%sOA1y#*7_3w-Pm5O4<{%ou7}?)x>5j_$*jh39xvnG;y7zaP'
    'pLoU>Hg(h8><}1jZ?%hENE$1tfVm{XgH3slt<*NLt`q?&OK)=?N8o+OhO7mF9ON6HJ#|S6OX8$emgp?KxDel;)S~sfRo2+q;x~R~A2*beIL#ujm1I-}!yq?#'
    'T>0H`5oCoNnVW^ot0ekwCXAWWn5bp5)qUWYV%aIa=?2o7+r@zA(!9}BN%gPc`9%2>TOLM@ZTjWHH#PZ9peq0KfjyF7E(rO%VbTREdnoaKXyHe56}sm@OJaWO'
    '0znX9{#nXfWmRaIGsw%o!L$4+CDd~hD(G-O$0?#>0EF_w6ogsyPI3ppRdS9gx1V1`aFqakO$@3OKELi<#0ziU0b1MfOk}AFmc|LE9MP7gzR)`wj=-fKIC!L5'
    '0%_FGVi%Q5V_v=QR&<ZimEJachFC$7xa#4}^1K$)jl?Nwz4%qfnGCHO#d`oN!wsR^jHq_%74_~j>S>&rdqISsT6eBzGaf`yZ9`Qk&}C=`<=-Orq00(=gHcO+'
    '=h_3a#ai-it9x@Kg|NB-Bq1c_Ufo|8*-Ogzgdvg~))&i{ENn-_V57SqdR@jH+_SZ)$-!vi#qnNAy+g3Byot?!)kd%vj(@PT-1MVBlR9&mozw&aR3q^`g)t)a'
    '$kw!HMDX1t;mJ!z(BA-dQ8VTm|Dc|wyG+ohbmk4(&9RGeQx2xQBID%v7B!r_uZG?sULUN>VL5}M;CU{i76t{!-}$OvK=nMJK%<JsMGTvg+9nXD4ezDURBL!&'
    '?^+jRT2<3DnEeM)jI9e;m<PTlsDBKsoF|NGwAqKpH&ZbWrqwKQ)eBjpRm}ICjGQx+aF*JWGORV0vd<&1KbfGYp?rP!8s}ss(1MGg$d{!dY)q)ws(BX&9oDnQ'
    '<Wt=5>CVI(7NF2m=&T=}Gtio3V8Y}1r*s&EBv$iE4p4xw#krpIL~A7_ge!%t2ZTPZ>~9Wf9B@v69A>2|+CjmC1z5z~Qt!Zmb6jO*!y+R-^2+U%vLxp6*Lm}?'
    'y9KgCZD3EpXe^fp1yj8n2W!R#2UTuRw=>guxr}3?!VJ^2r@YB+kVX+jN4^%;XAvIk;qPAlE@an)J<Pl!m1KLLFskAAL>y3RCJ0$Oo3`#+pujB-0w)`Z&CQKL'
    'OHxmKZZ*mk8GqCZWE5pwBOGw*<e##5ZEndoR!3L}hR!*wokeud2ds<y;rt$V8_%sQUp;#!HwuEXd=XaE-~?^A5-U{|ZcE_6qwQC-0}6rQ*<pEW1eKEckXstk'
    '1SoYuSca?VobnE}HUV|u<FF$RoVU>{%y{LwO9mw?xW=`@d46#18y~RYg*<V)$9ap<FCsLVe^@D8u;<PcRv`?QFI5>9i6;QZZF))nUL#xJ$E6~83VQdZl+iId'
    'KS&8ykfV1->)Kk@e8fgQKMWdHA+|!Zpfs<OhQsk#9UY-(y5GRSZmXHG2MvUX>=eqW>T75Kq4WkF-4XtZpCyjW@kQ3TNLJy3G4t=bTeG+m*?H#L;>I}V<Le7t'
    'YbY;XU%OKM;p}75;l=y~&}<Np0ote#kD^*tZ%W;G?_%MSK?fpc*#Y69_cS?`QSlOBn~m?oe(ri^K0?BPLEwS&B&p%Bs=?BwZSZU7qc>`}qlOvr?zA49>zFAR'
    'JHe4MRF@xV(EL>l?|QyNoO|liHC=Uv`D+19l}r4T>O_AR?T7MS&@cy3r6(T~|CHv?@opXc{XeJ%Tn*6a3kC#~1q}p*`~RSt|K8*pr6T_i-XMCf)Nq^{D^!#w'
    'XBG;m^0h{HfQfGr-qnZXBAT69p(Kl^6ynkkeSbv9TS>V<NF(~Ec79wRc>NV?tL8pBv+zH@guhPJAhg<-L(CfZdg7gQNULU<Eb^bbTG<2bq;-qVon@eFRr6KP'
    'XNAq()tsDGDFr+36^4j09R2eU1chhXeu#Z)oNGUY$_yo4BxS?ETGWqN3`T;cYuD$%aa+f+X(pbMTD=?(^PWTHBpFASrlyD9ugk+M1qaT2<}kwdju*)in7Fr2'
    'OO$X#E9TYvC4&+J7k=!TchFwVtXl6&nwF9*9}h&otbmnI2sLg;)u^~p$TSnyqP>RYrHVy8S?y+lO+s?!*<nqTzQm9uLNsLKmB?K6mRSG|rEHaNov4&}p7Ukh'
    '^S#D_vrj)u`nIi$ZW*LuD2nV^MvHf1&-I-gX%n0<_Iiy3F&krU{yNgybu5z71(g3v?fbgV<QqW=zk6_$w3Dh6b5p|JM5I3J{UqA$$c(Sk;+L=Ec@0Y-%HFbY'
    'XZC(Fz~P3;xAlA5J7>0d1M&C7j04T&8o5iDo`qZ&%0YGfCDn5vRL@xD<lN685HIPNsycN5sbU4PE(gFOLbd@UBYjcy4px6EGVdOAS%Ir@T*Vb@T0)yzBY`w~'
    '*c0EEGNO<;z#Z|4lIVu=#&dd6uY4R2;p5aHYW!iKtEXpY|A7tGGx&7ZgkY#u)YZT(%&@5NezVz4-U5_4pN^H?GvFF{RROk5oW)0kMmWdlbfg<j%^qwLuo`jT'
    's6hFf&%x@iYQEh7YZ6UgC)HA4lVqWdQ%zCJjl|+W^48YJ$)&B*a@P{Vwj%{Wb_U}>e3~8A5AmnLh6}MWlBv$lO906R^;R?5W<m7rJFpWpuh}R<K1sDb&XbrF'
    'N8Bow;KDfawceHW<?2H4uUMxUI~-pMsfNEW=XdoH(h|tNg>>t0$Ih|_IjY^y?NMeicBjpF(a&Et0M?|Psq!7hI$@fI^it#eOV3B@U@=!`_4&{GFPvwqZjF$@'
    'J|K7JbN9WG2$UJJ8K%Y}rP^OnalQsvHCdB?;a)vBUiJncTl14>e!xn)sme7jAio*IW)RtnL<My--B&o>vOCC>M{Azvc7z^dJ_7XQLJrEv#exv|Rd-Bk5nEgW'
    'h5Yk(Wd^xK`mRGuE<2i5qSqL!Hp1Z5jgM8i`92gC5P0`zkc0MUU(OqG*$*4ITV5=-#Fxt)S6s8eoFG1_W{Fdn1vRA7Zu$|Jo0}UfTh%ugm{=X0SRzemwWkVu'
    'QQBG0<DA=jP^&^e2y{Z!O<g~-f`6534mR^sxT@7y-jw2qYhg$G$I~c0S`=8YL{UoEKa1YXW@kz5h2Q!A_Cux?2-hTya(wmotYfh){cYPC#5`R-L-bzn3$W*o'
    'nu~r2mptcffBKq<RE7`x>Zd-<4T*?(?P5q`ec5hyrhJd5N#Yp3<0^Y-{;ki6Zl-e2BlS!`Ea6;3fz#goGo!r@#@+Ug(9K<PUUTya^Y6+<&3Dnm3=RbJ0t*C0'
    '_|J@^p@X@dlOw&psi}jpslJo3uBn5$k*=e$zJsCJf1XORA%0)$+8Z+RYZ$MIGU_qx0;`{+g2%I&1=#SLAV!L>S&=CamKEBh{B)B%8N87R)V&EAY=7VpY-j0y'
    'vXoV&ZA!2ro#U5RZJ0e-2t#pd)<mf>Q_yxY#t(?~NjC$NSZQ;>&@!)UaQJxGKkM^kfGBqINeg&Z1;!QqnJzZ7OvKR!DRR5AAfW_TS6Sy!QdnoCXsR!e7h}E%'
    '$D(O(*jm57Fs)EFEH|)G9POB%X2r(|-cV7f;(oVGYXhSz8tS>Bxd}nbs)HsmNeu}Kj}bj3dlFSFt7M={C)djg>)eAnNCp?hD@pC7yB0w$sJt87npUyr!H!Dy'
    '(WalVO|B{Xx!=25y5E`-Q|rBgx)owdV`k1!=s0d<SXfIWEy3!+XTmfsI*~4EC{bTfh3qWh<-I!R)D@dnNoB&|tc53O3=%Q2Fs4c|!M`R2*6t{GVQF}I_=d3I'
    'wG29PD@iir!Sm4=isvhW>>&j<NkhqmzioD1w#c54t~Y@HtASy5?vI9UU6((LfJ(oF$7qQ86OAaL%RE8{OEbi?3UAhs7KbMIiJx_%i9Ie-2Zr~p!KL6Ijg-c='
    '>8@7^k{3ZhMS(>6kU-_%m<K_~(4BxR6T0r(?<-KQW44VkT!8oGf&r00xY01j+AS06zY3O@J5i4KO|0+(E0HOQvn_%;^Aa8v{;|Z<9xhEoq5SI^GgxBs2!+O?'
    'jzu(sl0tP}<0q97u1w>{U{>S!b}Qsle1BUZxdw$Yh?%JYunD+e7~1+3XcdnS9&tS}&F>*XRY9HS`shV3T7T)4;6|wmGI-DrEysV-64)}LtWf|!8T$A`ia%NO'
    'NiJQI>(MU@K6(@qt*f`AtOj9mn-#a@*Txs5Wm`KJ7HQ2L6~bS@GYgOQeaqU%G@za=@aPyR>6Vuk;aO)h#!FMWuoPM8iQ08rFUVQOMJ0b`bs+L=z2ka^qa9^h'
    'b@WVbBVov7E<$JT0NWPCM*!h|L;<Tc8SGzFGS|uIJax;u-o;eMeYucSLZUEPHx)7QvVxw^ZeP+gyh$@y$A&O@oA+rL0LUis4X)v{K$@>bx(t9{p}8@;$`zlW'
    'e;v+uv7oh){^j}7-zWT(Vz(&}ds4Qb!@iy-QTwXpsm+3S*6TbL!;|vK$FU7H9MtCt_yi^Wsw@g;NR`HLF1wVij+G1~7u9jK6UQfiT@BASH_?YXMF;n17m;l|'
    '$joe^Pak*@EbXq2NwB-bR{3mgk>17L@9H&)Xxx$+*h0Vl_`_dekdb{a>B*6b+j&{<b=I&Y4_TIQ=PE6TQq&y)Pd!In1y6lHpK>@VJys5-8U_~W9m>6=hYqOH'
    '594gbuFx;&<E6+O3UouSk=?uv5a2x@RK<wvQNExHMBXF>)L`RKj9k|dHcgE5G*!(*Z4PvvxJ0MGvJT`9uy6ZPUFzY(Zv9TqPIJDZ-{Z0)rb|BydD6K|9#ZTu'
    'Sa-LUE{=2@*U;=9^#+MTD#ZL{V{0%+3y+B|bYvGEnnJW1$Ch7;KDf|(#?S%8iiEgsMP%W8?1#j6(?oTExwGu(-8Ow`<l8fby->hbC}yplJZYt9TM%>QRSR=M'
    'z!wDg5E4lI^cnFbhZkx!IEGn4IEUPa0`=I0w~7oho@&ylq#=gw5Uz-k{61<i>F2W@6>SBB{GJm$Z8L8%BJJbny0jD$9D{6VGgtPg7$k<eiF>}~5#cb@EL`lt'
    '`An+oRS_+NWG^c8a}Ic%>-l%VB_Hv5;X(icI$-#ZBgy|1-2cl_U<!N9=0Nm*OLuNS4~aU?{Jb8XAI8-h0ojNgFr2LiEJmndq-Y==r-A?M3U(uRdEvd~J*J}k'
    'H$f)sw@Cuf1SFo(ing}uLVaaJy*PU(A~4MS2dS(ZA=7YzuZRSZin<aHgU_!|%~qUZEbgE0tF@4WY$>Tnf)w!1-8M>xk^0I41}XN*hyHp><~=vsb*qI+pit6Z'
    'u1G}^-cE%##udM29P)l@+`6L+ioP?B-&MQmA}FiBkDg{iEM)o`7{Y0^jL|`<#f;R`ueK%z`6S~o;un6=#}=DBn7+~wpdxHgs-Jye#OXS%bx?-P<TO)4#B^M?'
    'Z-NXJL^ND9R7_ajd>uVueP_?zXLOFX57WP@(XM7*MPq6u69n;Gv@J_-<z`R{^s|FpkAvs#99#+2$)Z4lIjQ7S*>t&Qh~Q-Nx)Nf5R7MzN%*e`mdQ;%?{S9Z4'
    'N+5O?{*%3FGIx@Y!WhgGowC`pCFbVi(-U)-QzP645G9>5Lx$hX<_3&(NpM`neDLDKCojZdB0vsMu?q~A5Fh+@Ey5617ebTL@#Q<_m#Ol(WNUnbcK!yue=?)K'
    '*NYB0oJW&?`y!BkF+7g&z9_+Jx7;9ncp^)S*36fZD^@(*ou1M>D`JXz4;rcf;}_5wGmK8iY#ZaSfL|lV_UZ13a9#pR!og5+-LC<E?4s#@yubD?Ki}{Bxqq(j'
    'KSL^FZfbN^Awx$<3_W-a{&GFJoxX^EecO!=+?gRpLl*Ey%29YT;0F~~LRb;#2kbeFg#?t@X=8l;g4O!19h{dM{Y9n={R3D9?vD;OIO_mn*`3fig(p|+C6kpC'
    '^CINIf6Dg7BfdP-!+azk*fwK0hZGp4xIT62L?y=bFWbcT0ZF#v&h*9d7|>gV5fdVqrTy??)PVOps^hy&A-4jqlkP_@<i>pRp93?K7FfrpH09XU=Qz95@nJ$I'
    'FI<<I7Ddf0gc~U119?F*uv^Co2K7Y{=)=Xe1X*GBm<C}FLl&OrcJ^4%qlFOQfc_ZxHoJrrV8D}3xn<4-g8;200tE@FhLTI6D{bXF%?So&gctg=b?>x^<H+}@'
    '>BstUaOPTj8RbhNZm8&k3%g~<d5ItgnZN)9pgr3X^MgvOBPyJ5|H~5%f>SUQDc?z+VPvi`NlD|hdZ)X_By)gOhgi&bJnq&NnN|BGY!`5f&hD{cD*(j=3}Dia'
    'MT0Z3OK{tu)gCrTKw;JvDRb<7-P0Iq1!HI(ZVciyBT`>s=>41$1c1J24~^5su0j_Ai2zxywJT6kP{&%Cs$hXJQ{^g!bCw~hz~sjk4mryvqWq9|j-?N@XvTrL'
    'm|d}?k6>*GyTK!koGUR9Pm7>{o)!nTvYL9guO$cF5^AiA*Dt6+be1OB9w+rSfS$N#=e%GPXsC04UDQpe5FX3!`zUQvJ6Z;gr*~3vSoQTj<mHF=CJEb|n5Vt<'
    '-*(b1v(ebijSn{|-?cH(pz7L^luoe7-cQnD&c#dsfwI2hGo+ftjjoaZLOE<#Gc|A1XdrmC0Kemnj@MKlrXs_d<PguEI)SsHj-q<Af%Od%^;cm>^1JiHSa`<P'
    '=r9C0C~b$efxBffua>x!+^h9WGR5%=m_RL)wM@)8FKo7*+*Osyt>f6c%Nb19G$^pRjxM}LpXRDq2X!BmLb{AR9$K*eQGjLJ8qqs9X)Z0AOF;<ZlN@*VprnOI'
    '0MRtGjb(X^R)PbO887^ORR<@pr;06}EhMY)$!nG*0ublyXd^Mc1tDD@`f^U&ba#8i;=xTvm{py*LRe#e6#0htrpM}M^AoH&$G{b)EZY`C44~eWh3;HeX>R0-'
    'JfW>TYSW#tfH14a8l<gdM7T^3>8d5A;yObATdK7jtH7%pAt%u2uVivzb_J>vi^+wKa0S=5L9SkE-|Dw9y|mQf_*N*jR<u2N0yd!s{O|-#OJPrAC;<q<ph@74'
    '6VVU!#;oUfBFcRO!6C7OU7fC_HC?G6EzG9d2qhG3Y%O0u6?y&xbBU2-R{?Ar?tvJXVlOXM+hYw3;+`k{R{we!Vckb}1{2=THkmD`{qO_w+=5HorBH68Yz!fs'
    ';L$$WoHsdb>fB!;SaN2;^{0Sjy4)su8PtQXIE)Zy&tR;>Z@@1YO>28?(*P{N$B0y*lYd#t)>tnt)}Y~ulp}-v7IDK3;QUOn;SfHBeh=n1h#UK7!{SzjO3~3('
    'NkkwPlbsfzbFvQ`3Ac8BoB_^F6C2^r<U5Z$0FP152IS6CPnA#N!IL{etfDC?y1ui40+-{~b(5Qe&BEZ<vrY0C^pQ@?-%#*3*%%cv+tcEP_v8Bg{iNcCaBBEg'
    'IxicvCrjuN$RYG2;8V0wyBVE>YV3sj!jP{1paHEFI63!bVMZSJAtCO-|C;#cS<EardZvf3yL6Za@8T|YzYh-DNC6RR$QRD;*o<VpJ~gKub(T=dYJn6I6wABV'
    'f4d9Xy*BfCEU#)b)V$9y+?9}_JyP_H5`fohrde2B$Pqkls$5w`=cw5jtja3VPkY`>R~BH!Jk0!zcG2~d;h9a))PEgx6~1;-(Bfw9%<-bW2zx{y`f!!$_ec+{'
    'M_i@+u6q1?3ZSD&FyJq6Gzccv)hgS8{?w%I4%Hnq#FlU(1r<sfT$28nqBmQ2yaM|pu)Ej|j|6EDxo>y;*e?e0%$uhvT+wcN!;kWKLyvb$N6B!E?ipgARug5m'
    '?t<q6<kK47E!|7;YFk9e_;>aR#ZS3Rk0+2+rbvO7VV3|zRG&GZ$sl<O^7oEqP^9=Ao%0gbo&#jKF;og(gA~&ZU)5Wr7~-1Go;?*EW18Q0t+qdUG=x~9CFReg'
    'h6W|+oK?YT^isYk^S!YF3|dTW6psg`VG?$ac5TJ{i%52eF)m-~w%>tuR&Z~>$#(JbjKu_&QYEHTMrV2XDBH>W^a#R@IHz{8q`=dS#<1JdO$p6Y!7_P}ORs(D'
    'PrE13z`);(k5sOLd@`T$_~UV=zUDGA9LuQvn+$fCQgp&z2&N$YC+dYSPt_7HhIe>Z%LT(F^s)E!&Qs9G@?4}VF;jajRVFPJ_CH%v<?A)~Vj*3i+6>B)R~AFD'
    '1Ze}PiL;ts&2Gmv1>AM2wBuxduCY*(+SyoWs-)V`X{xfqq`2UgFStRs3Jy4u=bB4FVbgkKZs{88*UlRzPio8Al!3q{5@*YxH9Cg3<{NAVO#lGx<h<17CAbdt'
    'pRo?LkFpaF<(z6`>xAbp4Qc;o*2(cZJH*^hs_`!AK_GZR^s=z30be*RbDBbVIx&$-0|c8npu8bV&Hd>S(i0+Kl`@4O5YWF<4QFAc_(hVZ)MhR$?5h_RLz7GF'
    'QMiO=Gv?tHh-vc23Ne=^);4pQ2v}^3ztaz0V?_s4cq=_kq{B#}CqqJ3rTw!?oov(6&np<M%irO!Y0{?tJYZU?$hg>P`8I?2aB_fG;Iy%2Y?7xYCh=Q^q;B_S'
    '>z!P%_~%A)qZ21|#Voa<A?!9}QDf~sVsKa!bj?FoIdb)x!CN7{7XK#KD1}goQCkv!Ks4~lm9o<Xz5#i6n9OBNE=1UcAP(tQCr8C)$RJ^4LgjV0!M0XwU6|kf'
    'I@$Tr5WnI)a*5!4tIOBPBO^wmAF2!&k`!I|N+P+Y9fEFTjfbP5ca=Y0J*EW776cSU2bs}2NOXjCJd`#Sm_+`TJA92RdQS|KjaWRamNxVBl!}nc2cyOyzZM1='
    ';`8*3VU7gDB_BuHG*469CsCq*x2a-q^uxZzYQZjGa8z2$#%I3Es8oY5c6WG>l79}GvuW8%;PO~y;*}do%N+E=eDF1Wt2C~+&>7zSaTbHO7WYln`I3GrK{cA5'
    '0#KNI*K}N3p0uz?T1AyEr~pnc^!`}r$mn`pshRvR&G2ROKDlZbyoLK8lx&1_d)YzEZy``W6;L#7@?g1J>v7Z$>R<9MK!S0rr;K`?)VS66D+VwPfIxtoXE&v$'
    'q-}f8p@?sXt45_KzUB13l;;+(D-|WC>_<xokonQCra7x{Y$l>XYb*IKm_b*4I1=oZf!2m^H5*oXk<zo+tu{)?-$wbFdbSw1TqpQ@Rft$;yCeD4`1P(tJO|V?'
    'U#$JIF2110ZMoKsl4+w?Xc0J*&}M_x1L@T1qPG>O+Mr(#lyjmV&BVg_mfMdwpDE9bu_5)a$~iyZK{0Q9SE}T++vX!ar_|C#c05hJZJ~%xMnF})kVzIAUG!U)'
    'j+*k8p?&_<uq#!pC{tM=X=oKtCmg3^(Y@7J>Yw@MJ)ci)L|^`_F~aWbQez>`<0aEjxryEmlrU6zzKN~b8t#OAF7<bzk8?n3gm21gi|0BGhcvPU>d<N!m$~#!'
    'vYlPuO{Qp##m>-WPRCP?n=|>9^^LJ=Shi^_DMspVZi2l!@eQnOu5G<WoMs9j1X*fjTlr1vh++7t0cI&&-|kpmu*RiIWJ-)1cvq(wD2t8=K}jc|0rRa<{&*i*'
    'xfY-JYL+f}+!lSSRPHLbykt3XvinN;?$#&T=ht2J`ArYUZd%q)gM4YWjQgL4&Tf!+l0djrl*PJsX0Ffk(8jrAp-N@LU!L?BVxjMN+9$M{NjR$?d<!VJtFP+9'
    'tH&yMbdS4_eNYCkg1PQquZN%h%d;B>yZY`x<YRjk2#BU+cpF^5MNbV`fJW7MtcJ8?ITrJXzZ#Yn!DkNlf7Rx1hr9OLYfkq8Xwa>|*$XtYQX~o(Z>PYp#V4)J'
    'JPeguy$$OuJ3&^;+7u>Gxh)idnb(777K%8acR9p_m*xe!Rr$jd_dKQY*ie>W6W0yTCycniX@Rj=;hWx#f>VTNCR@V)5HB70s?#8>Y;=n^q0owi-Vi0WMUx|X'
    'TrXUz)IoXb`zvJ5W89}FG?ZV-Bj}D|IhM57v71)j^yF)>Og~YHG-EppG&L573C^5EPSVU&*-n~93~$=P(;UH<>ekiF_yW&(+Wv8sm&45SFEeu6)O7iEQnOxC'
    'F%X7Z9|BL=y0n<dHcx>psyfg0C$S<jw&Xa}5joD=WonV)hMk-eBIi&1Xdi0yIU0?}9Z;m=2^TED!APXDV)wR~6Wf_6-H(^YeQfk$6g@knZt8)SO6NQFV6eEB'
    'YF_~BQTN{q?qAW_2O<B2MVS9_H}jtbw}Z2duBE=Isg?16IyKt4WvOX8ZnUC#uhj70Av5KftubG_uue?3wq3fe%P?(Q;ir%3=^-cnQ3QkWZB|zO={f=06<HpV'
    'e9}W9obGG$JQ)C{Dq^>mQTjEyae<)9zVNUXxp~{(YJou`uX!KbX?<mTABR~X(p4)(Y}r4y^JX9qFbHH+SfVO&Qm|+uj!deuZqkb!RW4{G>#R;XLRU!}E6iAk'
    '2(~0?zw3!C?=Wq!s;YdK_t-E?8esdHZ1?$$v655=^f(S&)yKDNHRtGaq-}b>KL#1uAo`itrq*#R^_QOO2-`!e_$C~0yuAQUW#q-EElC=qOaSe;dBGgnue>Vd'
    'h)EKT>JC$bh9r$n0_U6c7G#dW+oXfmCdr@s_cJ=HhZ=0lg@~NL&^Mu*XY2Ju?Xu`S(7vP?>Z>Z(lDk6iCRiUJ%*3@2VICNm)1<p~wzPD;KJU3XzJ=c|<EO`H'
    'yQHCxPMBSP%bC7<`psaRSuwkPoE?N*=;}8`EMw)hl}oiTm=;yKIUlousxdTBsuh`@@M+hF7_en%Gfk;)uP1p*&zVAdVC0hqxtG!m!6a8KLFb~PZ8{Vu)jOty'
    'x&9`)1)c1Kp$VgwN|Vxw;-|E%;7aAC^rE-`ClxmD_N@cM{eHfVKa`5w!Gh#dDg)Q6A>Uz0*_ZdV{%l4TBA#pO<9ejAUS<illP)nUL`v?@eMlvjwS0kzC_V)|'
    'NqWre-}Jo!qg5W$AZ<DRa)ZkrxKEY-3IFN(sVChd^MmfIKSyT2X<5?37U;e&N4sfK;i5hxc+!TWb247L@PU@Y!HzMMpI5vefjHY08+hP`dV)IAtR@AkHBlpK'
    'nlTqtvdmb9XC|7!1oK_?s|8csz&_GbFF{2PnrP?H_sq9JJFMLwE{-a)r;e&Pr~HlG5Hn6vB=1WN(h9vTu;{-`X92WUjEpXWZ;M$RcaA$RhQnRg_vYjb=~KW@'
    'yN0m^@;z9<l3d2k{)jvrj%KAWZvv?ONPALu&qfsW3n0Nh^)=vdC-I8}^HXgD0a1g*5QPcO)%$d$fnG(%6R!i2KBOUdMI7%77sCnTQUzDi5Y^O_{SiEH6%$%#'
    '^L0nE_u=O-nJEqCLSXI#Esec1+Nw^qkanrlRy^2oeG8|zFy9a~;v*vs3{*v|sHBOvCF0*fpn-L;^WD7HRF~+Oj)l_Iumu)y&b_u0KHv>{1<wuy{o%ys9BUw$'
    '|BdXV344Z#*)w21-(VKQmBV=a#&a<o5TCU`ehzJ?HKV6=xceSF%fkLL!b28vs4`?<x4r5|C5=u5AiZOr>HNav3J}wh$gx{Hz8u5b(NOXo^XRKQCq^mRtQEz2'
    '!yLllc(U7SrYf4ZL09L1Dsq=E+X(G0W=u}Bg=^0bsV=@%ECye|AIo97ovX%fw-4W2O(<em#_Ox@*Wbw5Gdq(_dwWeJV$s7Y9$2s`J#B^7+XFcB!zG$pZkqTZ'
    'J>Q89Xp%s)7>4qWwyxfsAFliFwZYW}1<Cz2;3atBMiM1=5eQP1AW#Rb3a2DaX;T7Wl13fDAz*~z&Fgi=>qU5U_uJ8faL2D+sa=&ST18;U2^;<8Ty{I5U4zIq'
    '^q5os<_oxV_FX{O<rEeE&|$k+(eW|4+4gxISxR}iA>0er%Yo>{7Jp)aJkex?m7Ir!yV1@?USupEy92po<ndO6r1AkG%N~w!SxPPSNb-dPC#CPeL>k&Y&|8Tw'
    'ID!|q$V*zV<omW~t$v66#j!a}mT0Ybhs$)pbiNb)eg+zzBLe<>O5U7KSi^D#isX)4`IzvLVJdTJGmsFw;sKClq1FZYftu=fHjkeA^+x{|x>7y)wLueB*oj;q'
    'lQR}8xH+N45ugu`nZNsh1s4jV3Tl?-*b3O2r0T-#9kP`1<7s47`yIC+&3mCwf9zDu5tcZ5(^gqKw|&*WYh$iYu@f@4BLa@gw~6ob^$EG>2H$|KP&HQN8;(4P'
    '=Q|q+`!#6;6~;1+i~-MLITNot`W9Cm>mEiWF3laL43p?Z1C@>D8a&(02Nz<1;9vE?2)f7`H^TqezyBv~po?vY<nuPk{3kDs4;AYfChW#iY{X!uKuqJTH@%vM'
    'b&ml@-x_y=$XB$93uDXJy^&{I8yR1Z`zO);&=}wEdJ!NcIt&gPAZ)tHX6pRcPv`wrc~|dV${i)$gz&B$F&${c-vW8DFYmZzmB*0#$C-FaxuIDjR8L%RxDm|('
    '%AUg=iMJ^t(6YpKw?Auv{px%E{HVB_;<o1RyMGrpg04}PPRt5bFGVN9_m%R|slSWW;h`n#6`@RzNAH7&)phMLAwvUvw7^|N8m{(EJwIQmXAB2EFkFQaz74tC'
    'LjPt!Y(x>O^wJX7M=!7+$*N5n+vVewqOhG_<{Epf#_c>tpGsge?wx-=2wzUp=G?})^A@`qaHe>FEb{!;qo^o|Lf7>a!ram7iSFo-6&v4SVO@FJ{XB@wyPfM_'
    ')%xgvG-&?OWM<hODq!gXDa~p@ca@2=-*TT3BZ|x?K;^W{W6;rQ5~qY1;1j}ULu#}GN4YJ#@;Z*NWx(`par;wd!%~Q>@bh?1vwcx`8wkr@ui63kQIVmInW~Dz'
    '-><g-F|VXjr5-JanV#nr91?5>9=SL)?v*>#Q1zjfp>rk@0q^Bvn=klU6R&0*iAHToi!V{!Nr{O-j09BHaiC{|h8T2KbB~C1ccWq<A6dzp3vP(%I#!Jn7j!J0'
    'h?{>Tip1OD_%)>dYULWtw6*o-yUs|EU5OE-Gut6jSVHYBaAI>Xf0nF`@t4vyy4K~B2S3ekyKBrV<IKaUivj*wcJ)ih9tAGvS+wZ(oi(>txD|-6Tn=bsk~J?<'
    '--91Z2#QJcWDh&aHS>SIj_%|NzE1fkbo~clsQ!Nd^IzV}|1w<u7tL%r{>u+$Mb~dYsx&a)nB7K8<}Fq#Y1y&t%6?P&gb5E;+8=qEv=5knjVtHpMbk4juo8h{'
    'DL}@zBjW4x${+LKqFK~^Y<4N0I@nu~K;w(shriJ(nT1!jSiC-x9d^sHOL)YVqNQl`dKPZxnl3flQ!uAgi=mNAf?>EoFdQ05p|L$F5^=CXR6NQ|8%Bl|qX|2z'
    'owv=bRH;()M=Y$}LuEqcp<cJCB?=Wwv3y7UCFbWyrcdfv)}~Y^`Lmh(89MpPO=A>RWhEtrowBKh_z$WY0E;4?O1@f+MV6mJ#q#OtwFW6O9y1J3gO<RRJ}hpI'
    'g#|dc8_iM+%S@ska}^-dM}SM{1o_^!`$e6NWT+W=sc#`6C+oARx4b)%ZmtkN{~BK@%k})SLg3BaljhU<cHk~seWE+fm!qr0p_3%BZI(1UBv=o=6iG`u6J~%q'
    'c^Pw~>+fe>wnScIW$6IPlB=?O4xNAf=3uXz;(ttvqAYUZnP3erif*}Y1%ZW^vJ;XuXU5Xg{&~A$diyx$#FhTz#@h4y$IDw^E9Te1os(oU9EC+(f`V+!PFDPf'
    'Cs$TBvP73dAif+x+z<}mtF`mh^_vE&VUGtSEuZdH5D%@h_-HN<g)NAToCWi7N_`G%SG+HBl!Nlb!DPXh2Z!i2jnFjx$gXuyCNf9q5Y?Xtq{RF=mqf*s_~eOr'
    'sEUr<6qAYkP<EX>q*a&=T!H5_#L$R%lr=UDJKkyt4OO<C8N-0)@hzqqCLyqSSAmwu*Sm!IHwUe!W(yipl%$+X1e^p-duf}EV{&(d{K-5#lS_jKHRX|}`-D{W'
    '${7nMuXqL7Q4@U{3uR6zOb3Zw+UsO<q58p{%~^}D)8P$)J-@?g{!KhC13sdzp!3Qjx-hWxT(~*8@_TY-H!m5V3~+Wha=#cf(M{Rp!e3L<!>u(cH*Q|Xv6FC8'
    '(OycoVgq5qu#obrd0x?cO4niYwC&O4A+LF43c{;p*>a=U(kI8g?n3Nz;Qm@5>?taXQfiLMC7Ftr^GO;|Y!}$w24HboIC?}D^hGM0ObrEa{6*JHJL5gx`+F@{'
    'j>lH|MyoX~NumM}U8tSDda{XY>^{o{YOXHR)42)n{w#!wVz7|_oxM759O!8xNxM#CoJi?%?pK_*-wPh8G%|x(+FbBO>c+G6j9zcAkn@J?I<$sxeW!5YwU8cG'
    'p9>c-S;0iENr2GTbxjmRSe<pqfP-v<ike3OxB-B7SS6Y*qAXL>ddkvuOGwQ)Lc>!Vr%|e80g>GPU4xU0O6n}7-Ho9i0pxy3f!M|^84WOxYNXj5PR<_@?7UYY'
    '7Vmy$ZVRh&aAVWINHraYGKcx7ElA!`Bg0EJsi*fK#rd1oW$37${6D^C6zvwb323lUDHzh`1w|7e9M7*InEI!<wez)?9#n*!&0T|5!COWX&K$y)BuKZ|`pcnK'
    'WqN1t&+u&EUxmug(m9MB_Z4|n25oaDp&@ZxgNB++N!XR6jU!@&KAnaa>KybU!*Yff2Cir?-YN`#YkBJDq;pMGsIg;mGJhKH{O-21C;i9fRDmyhfrO5*L-AA%'
    'inaImk<}G&FvV;r^!fgh{~S+=p-k&Qq<SMf<Mi;`uP$#bWb4N19ux3&^`!lP?5S2S)cDVTuz{N5{C*US_W;~f6&R3^lrG2iO4rh5h^Oem*BP)+A6hS!mU^HC'
    'z;gyN@M0m{yZ8?`lSBFe-IoQ;DO7L0e+;Q<VHTzynb78iX2hEyV&TZYu%(W?KzSddaTIK<p1R1IZF2Xwa4Wt*YGhGq5eLnFxa&sh+<<YbW`ltZ1llH81WT8^'
    '6SN~XorxUp$jXAwqQSs>4F@=2GBBGt#bUZ}*gA4$7~~itTD<)ByHC+bTa2NFb9pshujx`JGg<J((U#Xq$1?7?>sM9`Fx0vj%25Z5Mp3?}bouA%)_n7`Oz5c!'
    'rjoWB|AB*DqXIyj<r!LYgW4Oj)&*lVxx{`(MkBz2-~&<gGM;giqxZff69v2%<G8{FtE_}L7F-m?GXAp}u)6p!D=b>~(z(?iKi|=wI%A{Pyt2~)OfD!=vkUIM'
    '9`<_ogr+6%N8IwUlzH5cIaG7wk<Ncld_6m5_6c@5UJ@4>s`6Rj9_`@<P;98qlb$xS@-yrfWPG|J2L^VwS214K<wSgYjHgLKT*2l>BLch+qg~x0l&+aY;=@d;'
    '$P9JjFoQx5grI)ES#(p>LPC74B#zuN?%8S^-5Twa+gyUR!Ib(5yXu2WYlVd1ncZy!^zvhDj>58Pjn|?X*4r3>T*g4v+l4TsETV{W%UyTv3acab;jOQBX-KyV'
    'kNC7P<<c;Zrb&YN$aF+L#p}S@G4z$-Cq~}lT|zA4D1=Yy#x)cNb$JY9NL7WFPB_2E3A#h1b>*$w&8tG<Zu>Iq+7j;T6;&j_rC4F^W2ysOepieKtRz3Q6Lfk`'
    'W19tI=GyclBnE=H)aP`=2n%KSJO}Jq<zz`lm<YnO#>r-iWe%a$knkzkjCwQ(`PKamPQCzgX3Lw)afgCf6B@>j`Ua3pZ~jiV!SWw(IZk-PYwfYweKo1<ry+p#'
    'v3Irnqzj7*ycK9A(Dh<WVVcI39f-PEdcOTScjtaQUqTV~`Evxwr~#|efPjA>nWFRhQd0!xaL%qHu$D==_`OShbCl2OqP4~@KfM_usHh%zNK9gcYe&v919+m1'
    'D?_Z^9thT9r+q3wrlDLlS4H~mkV6%79n=}1zB_vBl(YC83O9F=afDrracHx<*#hk?aQu8bHx`VD?~)yq@{1e6!vx5%#DVXz4NILaaY}K*>IcgO?dd=+wWCJ2'
    '#+R50?6}|+sy-fZ;R?X`g^ywoAA-p9kfM<wQSbx$?{p*Hfu}730R#k#3<QMp-{{80*umVy?LWF9bpJt}fAyZJ{YURvwr<~f!Xi~PUgeO=B4}_G_`ANQJnV-7'
    'CdjWWt7@jQxWp{0Ri7O^-A4m%SzmBCgM^We6Z~zQ4AL=zP9(IJUJjb2ZtGsbAtwr~9S5Q)<9_Vr(yB4_wt#!fionLmiEu30{mP|3Ve`;rQ?wHC85$M5G~w+H'
    'S;6^w20PR;6ovJhVOyJlP4l?>BYaWucMsVcS*i>BC8AdO(KC&Ef1k_!hbJZF9kqLfU^Fg){lLdGSpKBq_4qunnL=(>62Mgdl1yn{wQm3wwpSFuW1l-ntUS7O'
    'o>tP1P`EXDi(`jy5Pg?AETT-R9CCT9h>z{+(0>&&U7aU#JYjCZP8Bw_zo~x<V=)(zAl6@|)Ja>kZs1OSox-vvTayN@frdd~16MDX9KB*Xv-0}Tj(5}csMSb3'
    '1{kA9{_-D<P{s$xzgFh27{^1lg?CkHBAkgnlJ2iVLQa;5XS7*`FDL?@!T7oitdry;9){p6Dz$iK7M;Za1L1m_?`tkq4<PRki|o>XXjTChuQ!owKpQqHAJC|u'
    'R*Q;A2VhxOofS3|w`~fxkIp=K4xp>E<t}Pr>ZRi2dCqE_vAmHQ2FFg7TjIz&NMe)_r%UqO!zo1JU%v?n=QPyZwpzV<|F6o<0w|7dUDvn;cM0z9F2UV3xa;8V'
    '4grF@1$TFMC)hyH;O_1gZuWofIXRp2?|tr_sp(nWRo}CEs;0jFdi7fGtKv{JcZn|p=pYO(h%Vph$>=L{4LERe^{Shu3WBM(R@8ORkP*-N(YN7E#;|scX`K$='
    'daOVZJr+lCV-faquJy%}#xJUPee_nTmdgm<;EEMIN`|z_!vtF26(VHA$=`>-YaoB%Ok^fkIHZ1_LWA)F9ov+oTvRuRV>~G-yey1XTpC<Qrb`Hfe&OTv7W2+7'
    '^j%Zx)c(z}=FN<INF_BBa*U1!S(C|EDv9m|e_!{Fu)^|KSULf6LC85YKH~5de>sX)6f9jw*#X_vOt|O5TXw!~E(}@K&qg6WnT~t6e4tZm9|Kui2bm|q5DS`v'
    'O*$R7Za87|A$M(A-6M1*@dX`Ko$yzrs0nXW276w{sabe)W*kU(&@k^P@F5q;_%ho|dLH+kWg!`#6e+Wrp@>Tk8f3x_dU)@Uj-ELUQpoY#6IQ#Z@ND4?K=i0)'
    '$QE#B)eIQtwhVQuzHCKCoi=Rg(=6ygYU#JKcwNlh3ZfeZXgzC{N8fytZp})(ylsi}l8YvGhK_M~Or3!u_@v2Sb8jucoEU`wDYsDGSk7FrPx2AY+bYL#L5Z7W'
    'Y7cOiKAsTOr|Ik?oV-NttE(CVo&DB-E;%KmMM8}3K?cTz4zdk36PJ#{m<)g)_ckHBfnU&msRD4n{v^Wjl4|MI>qGJliN)YktRMV}=4_|B{e#=98(#eqE=~y-'
    'V|5D#%qEj;<_8uR&ld_Y*w9FJ?;vmk7NU;~&bt%U8vRlyXP<q)(cLN&6=~#C&>DVibpttRSk6Py&qd)aEb~pgFo~f$+fENm`dFOnLLo~}HXoiCX@W@Mdmysp'
    'Bg9Y8_cad-dU=`2LIv2G-)q?|o0V!C(>ho5z!_7DA|Vsiv8Wf?q;3@T=KWEQq?0jTiB&HhV41Ju(HMDaa6rdQ5=n@<byzFdvkBRk0yy&{ILqAHXZw;i*eq&X'
    'qm0`%2eRM!bU5h<dNijl+qkk|+K83RWgpqFA^ypYTFG1zD3<d~AoD%=ikA~{>x8>C5}r~)3wWu1Q_^Reat4brTE_@g1sF6F3#4w+O(%0|>WSNNA%t*4p(^5)'
    '2kV$EE0L<mg6|sHqRNqp<dWPL%d5OKfG<Y|&i<ssvTtvh!dXw<Ja&TBYY+G-A|i0P6nZgCZwE(aqE4u6NRbtsrl0TaC1v}KVwQb4=xF79-dUgcE3}rr>@ar$'
    '<MbPW3gPn5eU=h&s&I77+$iMG#{q2pOu3?j!vN8l!522Smzx%yEjl5ElR-Yi&$a<1M@xOL+JXbVPdAstjBDfwelEhKd4qhETqs1QoXT2fH8L#FaE8t^6bbWA'
    'cj_X*dzIpYG*tirE*9*#`Pco}r^E7LGdDq%?R6rl=gRuz`tyCa5hTP^^`x>iu>2#!wJglJu{=h`Dtg~%3P-|Vm|fKYnanN?{mEKZFHXMVEx$1?Ou1C@=d@v0'
    'EvWBTsT{3dw<|Bu|9tgvooPI6Lj(c&!~z2H`wQeRRr3Fuf;Wf<aN2IVZ1;ow?wiYiO4^zxza^U1`jk0~Kb%tg?IvhAC|6R_XhH)N7_+kCcYX#&BqW_TmC|ra'
    '#sSN47vclfLIh)Ll9@!uph7{g)e+ukz^UozM}2b=s7B7q|F(HLXU$BBe9vMYleSiOOt?OMvCPJhM+6f+Uq^$ze=N?aY(H6-S5*L&S~pWWS*DDS-^$!tYFrxZ'
    'r7#=h^^#I1eXld%2>{MtB%0HHx!%7E@~x_O+#Xs_o>IAEkhw|JHG%W(u8y<MR9KQz+0Qc-)L>b<)5}9e&;6{a)xkD@K&PREd%URR4n2jXmI>!!0`ipx10=!Z'
    '__HJ7)m2LN!{Ola@p1hzwKltEmLquL4`+5$Q}VA|Q@&_y3a}GnEyyZG!IG`+$XA+nLM;UNPyGPR%6c6x<-4RG!vrm%b9A5%YgPwGH|P7D=Ea=dI^QnK>-DFq'
    'r`Gj(erX*LAOcVjgFqXDBqTUB?4}r}s<d>hN5SHqil$qFGx<M4OGrqRer8=t)R|SQLx{nCLH>oN=y9Wx#L=2MQ=&8#UTn#lH6r@~w!D><6az_<ME)c_jq|!N'
    'aF<cA*MCy8uQDKyaoX$t6>+hvLOPYhu>O2B*fuFyr3uRQW);>Dob;0ysuZ{$SWe`Q$aZ;w6Jbt|q@W{A<_|D>UbD|ouIiQ~i_|0}K%iXeM@<{5K{b`07Y{bh'
    'h>`;I@F5ftuoc>3PEYcEHsL%e8x!~;)uD2o0(?hO<X;s5&lO>uVDKz48Z{g6!hyQMm~ejZ=*TQcaaF?QfY_p)@($@U(EBTK4x^b2%S(pS{tv9rq6?L9&nq{{'
    'oPlq&sO+L?E`y-ei(eS4Inu<{CHHetOj*b*bMjaw;O>gad56e{Fj%c7W^u7+NZ1S#s3x!N>&6H!Dj}Xh>0uV%TE9#CP?(bUZ*)L-jVcE<W#Tv|TQLFgu|aB<'
    'wYtfVFaa96UXQ9rYA?D^vY<Xn6lyA9;3TG&q|;feWf0YxOrn^TTRXp)lP$S$G!+ZQ@DUZOjlNm82A+STPLcy1Ybj^BN@Gk8B^gl)M!Os82WD*W!EID}l8z{f'
    'y`Tqz(qW-BIp0vAY#>l#8Y6`}hzM??a695MGr$S_JRGoi(+I=svwBTbHcg2zhM^Fv%&z3*k&^+1Z89O$k_Cms$#iz~vxJf7h0&e+;P1bKKOux9zo9FLffY-M'
    'l5xMAAGID`Uwcs*GV`JcPZ&NMRfRY41K!Ngl{3Gljc?SrFqkKxrogR_mF$IqD_dQReo#G;#tV5M3@b(9GjYQ>>P2P&%|cY)X9UR5@vNc?&Nz%`!|6~Pv^a+*'
    '-QD@Oo*Tg}71%Zb+Qy2b{SWR9icp09D{GVIf>Q)<=oZ{V>NqKQD4y9AZq-P)4^VwYIj|KKEfoYpVxMnT?iz#APVa0f*%LW*x>8|ZZeO|!g1XGRNj%GKbUwhZ'
    '8-7xkeKk#cu@i3BAUuh>8am}l2RTHSwO&mOr`F7as6>e8N?RrZl_e%9KoDX8ANPE|mWUBl4Qu%r%*aNemB4>_-koXW`CZ8>Rq4fk4U4oAmkQ7qmvbWFO1lXV'
    'ek2;b@dxU%l}HI@1e6pDI2~|zn>N;W6xjO5S2<oNun8KOnl~`&yd4v)rW-f`+*{hkGTlaQjG!H{sq$K?#L_|uty}gr(;_IxiSppT^z?i-urN?&0y|gadO&kN'
    'Fld`rQH(&Ky9bYG+w^c{yt39-qXu!OdEk2N|52>D><Mb-ON8bcz~#GGvnwynPWPJjYJvte&PZezcR5phYlPfKU3P}_{Eew?DAjD9wLR#qEl1~?#|~ZJcuOeA'
    'CYR#mgP-6QTOP%|%CYRqRhWQ-tTgZz0z%YJUzR_Dp*Wz#J4(79t@{SNq+Cih!8q%x3{~4{gRr{i0)PRmy&psgtUkLe><jK%BoAi1E2Y;O*U}wUL>N#OQAxKj'
    '#WXE;!ct%_xCb+`Znw)VCz0VgmDtFv5)O~>dcdErg|HX1wE8}P_CQ^I)>*M@2-Cil)8!vuRw+U9LLgfSN2ymtAZs;-X`9R^JuAW#Adi{J;SK@+xv)5uD@eTB'
    'T)$tEn!xFxH${l4cOuHg+<}SHvml*9N8tpGtG%%Zg(1v^QT&C00bbYI&L6)*`<JCA80)xb)|PLWR_H6!++e;XV@D(e=F|g%6zQe9fdS0DADPPuM86)aZSpwg'
    'a6|5305e8kHMHGuKp6|w$k_@5I7Y245?z}p@iF$1k;>dA(NO`UJ1r-I*51?&q!l^j92K#?W0pGXKAN2wdNFmKkV>Bche-P<M-IWO&R24b1#8+XPn-?b7PRN#'
    '+h-ZG9$ZUQ<}*sje2Y;Z5I|jJj&r|SU$X&5IUOvKrN6upa5BD0jjlJ_9uX$lTBsNH3{9A<M|pG-H{bkz#1nng4a=zjK)5e5!?=|>S~lZz+Vc2-2?eu5cL0*r'
    'WOwRN%t7slS?sXsB(|M@;D{n)onu~u-B&BltAwBI(co_AaZyT*DF0TZjSem~y8Sflw%w7m@wS17<j>ipRvf6MXpi8)cmR`HVnLNTeTUZGprU!n6Op8VUO}kp'
    '9{bfLlh&jk**B$hU4+|`4WfH0_9l^n<tJiv2ErC;v!@a;IS=I;1pGPVg)c7@B2}k#THBm%(W?21{?`$eCL9!Ic_fclAUcmc$b?@LDvw@=b%dV?lU7Fa4qJBc'
    '{R7jQoo8x{evszu5y)Cdst-R8q2xqF1>eZLxAR^82!}Smj2P-C!o7`h2Sj@$tyA5J?ME(;pPlKOXLG*BTm&ij5$8)<Lky&T@;iOoF`W(FJOpmpMXCJiS9&Ah'
    '@iCM0+OTsy`{NMJa^=IQ?UC~iodK^^$G@v_n-%k0_^(wWh9iq^Zt=yU(FhYqY187tQu}IYpSol9ETM<C>B4%KwO<~<S}ScgG!lkXWPd)`vk*!EB8G9$V|*j<'
    'zXhtP5eLS3Lh>Nt&*3UpN}d5$RC|<o?Q}ALQR7?eWZ>JRox2>ORiMErFe;7ybtE$5X^0=xlvXpM#P~#Tig$hjeH=3z>1o(HkVtMJOUIM(`jyT3q<1}TB~8rA'
    'tV~GEMlmzGz+D|{Oq=QW=RA^4c>K<AElzBDy_uR*a|$Z-{B1-~Cg5(OqtwW|LxkTzyM#&AHW+l<37U|2^;tC{w_ra<of=2@C}gI?&1z*?PMf?)4&Ed>l!R0w'
    '%0~I<2fDc=-3{2FwNS?qZ?W^<OkN`6*dz3iY}9`z3|@d*q-U_xN)!^w+|*O?BOJOSw}3Xn&%`}Eu*PJrC@+##ScFFUAz|kyKejKxQw+L++TI5v5(eiZps20$'
    'l@cHEnXY;mMIjvI9;SUYj@rvl4rw+3r9D5eMOU_5>MlUwp(Eij;81wXZk)bvL;`2NL&e9S#*sBopbqC5Ij2`?(Do@lMx@c4hBhgBOf*=Y5IHI5Lw={$#aCDC'
    'C#vD$Fg9DY2&f0`=`O{_C#)gKBfHV`#lu1FcC4viT@_-1&=JuF4|{Sl%c<<RGe808WT~ckg3Axb7Dc_RDId_Ys*omwC+M73K4psJKbvzogP-KIG-)3BBV-XM'
    'LyQNx?!&3hxU#k67-F|{Rj;4R+2;VJ5gW`B;=c7WU9^`7-S3?H1*QpJ69VW((1`p;!)r5VE}x`^UO_2%5}NwrBZagDAXhaBx|Y6^^_*5$&A_zH^kSZ@sQ)1N'
    'fvs}6=$LYi2_ZdH!Q}OY%6M!7v<zGs9?oI&^mVI=dgog#j6!CuVrX1)#<^BJ6FN!f?hKq%QgFbN?`ZUT$Kx3D!_Q0i8z)al5+m^NjT~QHzcCv>JogD2dM>wD'
    ';d|GWE+x}0V~GGMrIon}(2KrKX*Q6LAIq;6=)Vg_&3_lxo(U{C1ywej3Fb#)zQ`_M@PY#yYsXbdU$}Dh$<?=j`b^`fO#_9F#l7cf8_(XnE3}><*`oWX9i*wI'
    '_Df2cD90b@8S4j+xN_DAv6@BF)rg@FRF|vPBM09d9O$*&mAB1>MYcF7NQ{`_H>ZORE|l&|nSCuGj)7`q|I(x^q`r63&T;iqF8qA4PZKSR7j+;)Tq%KehIHb<'
    '+vNKb_;45cE8F7Pb&IwaJY@+7BiP;YV<Nqd;RkP2+x1%iH14`Z6i7Q!j{WFMdAei!Q?IFGB<4dl>o67)o{%SPQx7%eg-xSmkzuMZcr}3J6En!Id)QG_dm;H?'
    'cLw};+@lZOp-;A1O~!^QY6ifj>oG1_#Id@wCMfphkkaZQ=o+)q$ZgR=!~begC<vQFq>|iUmk&+}47Lr|s?Jc~dAsgrHMTC=D=Zm$9UmwKO6CII^ed7bu(mN6'
    '1)3}35UChf><#zWf2!QPqdRq1d8)gi02-RB)1H6PBi<b<6nKHCXfW|ydw6R-WR2k0RI#A#W?sne!sVGgEc1oXkRE&^U7HEv{7EB*X({4^3ER%<PuxpEUX;fZ'
    '*Bq^&3fiR2)@WM#(D}s=^ytxu(}y;d=Np7NwZbcR&ciF7HnggUk&ciyyn5d8#`n3%!=qXK1YSwuRpjU02|T91>=}4HQMy~&Tp90vC8&tGjU`5KQ|0lhu>mt%'
    'Sin$eicW;i?#tnSrl&g>{U|CuKYb;Axg5D&?EH19yZF^|PaJ~^6deRzmFob~v2{iF`yoh(p}~@IG14|=`L6uaQ9qAnN|zoFvau1#dbQ?}jJ)0~_}<bd9hAi8'
    '6vh;=SIt<_l+KP<k8~$1CB2=+*2r2!Uurd#Z6O{}_;cL5FI_aBcwzuHUOC-aOWA=L$y+v>a3v_;CkE9%yG%Ove|@k#6eFg>lZC9i8H)=)-&ngzj&N~@f5~=@'
    'Z9nJOI)e<lBWdZ>BN~@;%*NwIM|49$e7O$j{q}14X7+1MzP;1`lTH0a!&^190e5uI2KTE9#N_BKw}h#^?Zh`WR_>>FPms}67e@tuH%YKTKrsIKF89yew@U2g'
    '=;fB%RsG&FMn@9byBuxwp|T!pj(P;3EVO2^&u0h-`e_bk)I##o+Et>rH7ED!!IU(iWg;QEtCjOFO6Zm7Xz?9y?rzT>O{r6((mOHtE91Z9@A^(g%qhMuyJ|aF'
    '!n}|H%&t}|b`(0u&b6<lKF*AqG@_;V=!>)2r5c5z`D;kb%nGL`R=8&JouuyVFb)uXs-j~C-)}_KCe}O@O$`PN(k7NCJw}hxGb&#xs7D0iD9L;oYb4Gy(nj<i'
    '<L9-S0ShV;Wy>MCEmBXCr&L0h<0z|wDa@E}h){uZm#b7H*2I%hR-o9FwGo1x=;sXUq#fk(Rv$n4*b3#_E0%NXj|KXDmge5z&YZ^7f7xGp1cl8keMRo0sv1_u'
    'tR(}7tP%j6xEVBr(0vV8Ah6+WT6_*q=?diFQ)%bN0KAK`ozP@Z=hcY1oYhO1#V<YBvxJ_7io#Zfr@`JvrXR8t!q^ERd5?li)Ros}9}$xpyK+~jaj4<+Xj73m'
    '6gYl)d%azsrsxa!wCGPOG<UD~>G(W8iU@}e-|GuZ`|-V`%Z>nkak)TMSZLM<((-?krGQ*Vu5lqpoE7&xA+bJOZP`*@pdlvL4$75R&|ErMjJNjBY9)%arcF<g'
    '1r$Jg2(f^{a_VYHhRx{TR}eSuFqAQO0j-gbycn8zVQ84BInyg*M0@<`IO!pj^dgEjAqVKqsfT!!(eScyOX)<#e#ORp63X+aZLbgE-=3l69%OxSHqc{w0cQxD'
    '4TP|Mgf6tgkYKnaX2HwKr`svM_j^vEMN~IY?T;r$(2!5Iz!k--dx=}=2#ne30AYYq#3|@pTF~pxqB5zIn^ZkOFp1f$bs0u?Y3Lq_!Aa{5OaZUb+Ng(OoYlg<'
    '9H>-}uGzOtl7o>OnS7!Q2-FlD91lFajb>j7Os)tt#>1-pkS1)AKhW0LNckF$w>NMr#3sjHo*qA}GCVc}c6uO#*1m3+`TfbwbW4nflGTL0&!NtP?CtWSIwEW@'
    '2ERKzB|U}AFdH$E8k}p%7W3pKi9FLJ^^76t1_tlIb%8k#p5@aSeyc;zu+xSy^hx7UE)pjUy-92{>09!E$dlcu5swKfFmMFF7;&neM<^Is;_c3JTj*pve?x&}'
    'KP?9KeixWCBDm(d8J`l%cDZrXY%08Q?oi9OMrKgGySWE|{)KnN)!n=Z*9V(UHRE~~)l0(apBi|)P5Mh953JxTNpZ^dD*I04N+A*I>fBIDu*v8CqSDgszK<Dt'
    '+%GV_#qdp+PPpn9)z!ri-y5WPzLIrX*?DsJ^2$%xhg*4q4v}{P_-jv{6^Cthp%rLh<RDwIUy@?mnDbEI_GsFnAyKdPq`h!3d`@}*O_+sD5`KGQRH`M>B+=56'
    '&We$l@stQxV$?o0M5<QL79Gl!pXdRVrMApW`-dyLmX+v<fHov{Vjb+%K;Zh>hdl~HliX<`EiYovxYvWZ5XLrOhWvbd)fkUoFb>5rxnmM=W=xpY6JPznJB0*Q'
    '`@reaDR=(`gC`rGxIH-P`3M5v@?aqDN06GQ2g5jAuSIU68#>Qh@kMo!_F%;4{tHtpxVCE*nK^Ak&CmBxOA<7iFbifiE?%WCpj;^0c|U58!x7EX>*^wljdyiK'
    'I{H%~k@z-t&hWM+9)s(fNvX}F6SIGHqE9Q@+t8y0&Rw5WaLjDxg=uwbH9UW~I#98kVIL#Bm&LdwH@E1|=nZippR*Se{{alkldEXXEz~CHY8E;?q;}?t+6ozo'
    'q90weF@U$WV2UvPn)M(R;Yt^1*9>fFjlhypKwXAVPpD>MYar??8M|9TD3FH$zkVL@vt+9CiqDu0*X@R>#&x=3@6~>eF~PXQOx~|FdPMo~Ex~&a9Spp}W71yf'
    '`YiA^;=t2bSO|Om#V*ceua5JPm(oF)M1}K1OTfvOnYX=JTTokcvAB%V*qqMSvkY*CbY{s%6WUHBeqRumQeD%UuOXF8oodUNDWT9_X!UvixjEK#yy{ljog%a7'
    'psB;*`Ag#%k!kb@C_50C^y|EWnDfaLL1uTJ!a-UtAuD{ZhJp;28SHJ;cuw|S%lQ@eGq!f~6o?p~Ks(k?wxXpu{kI+9KJZjLLX9bzxbfI4nn>7pG!~ApB~3el'
    'ny;hTH!Jp?1xCEB!9lJT`4#m%t@5jh8WZ=hxd8OAZ$b0$t_*R_k5L1iy0=+GDI$nYFP&sU8x4+@1-l(G3E#RGVp0lmf``}|g18%+0fJPUTNtQ_N`aY**!CRy'
    '-ce8fQm?+JYt}u7{oj^rV>~xD$tOcyf>7KHC%PlSPEqK9m@g2eUMvUG;(DVDEl2RqB@V80zUCHZmf$}xCy@N0_}=0Z+k&tOBaNg_gAl$I)pTJw`>^S3)75Q%'
    '5f<#{yu7LFn)-2;>HZy(1sm-MA00S$&S2>jsfS9jgJVLLn9tDO|3Q&Z_%V7v<_@7oJEq2TV9JmZc?Kbb8yG5wxhwY}WMa-m4xmZ!rE%WnL&>j@(@Oq+^mO@;'
    'yNxSh^c#N9YyI2aw3@qt9Jo-_AmUhC7dI_m8xOwABO~O))=mYP+VeQ?szoe#6Ml=7-pDXO0)VN`3Y_#nvx*O%=}r<bPPu>EeW4)htkO=>wgdH;HkeqkZkZVn'
    '*t&JqkGGS2=A8wV!N)uC>vol!QZINjP4IdQ0P@iBmZLI~BJySEmVR6#(nESHI$m|Sm1E)dDUlnRs>LEdj!b~pH~nJQ|1b+1Crtemp_~aqDqWkea>QHzEutvl'
    'z4zxOg%Oqt1PDkdJO~KE-<BNzT@$HN<3BW!?yEOiuwY}=iUpzR$_WBF=;7G(fZSeqFj_oPqGU?>RFrztZ(XF+qs}JH1T`IGWasj3_p3f`xrw&uSO`mDHx3A%'
    'k7?d}WgvB4a#_I);S2#QDIMTfw3}+$S35L%ac-SmXj7s>G+JTQg<o|>)0vX*+cbKxQCw(~C^{?)_OwQdwCRR;)f#kMn6jolHN#KcD_R-8wPL>YeVLx`^uR7`'
    'P^HeQ(4l87Y2(7+){xDx7geeGKG4e)Ed-uLe9Hc@bWy)TjRgxMGO#~xX_(X@O3F&G-pb5-`74d_^*!k=@$fKDe<51KBrD#?&Rw(y{{k{f23u%odm2;!h!unh'
    '(1Eh(DO8B|ha*F^d%|T`Np9K-Y}ZP~`Z}L?N4u`w;_AfkZSh3^^K#h3!tod0yXRNJ=|G*UZLk}GL7GukMctOqbSxt%R?`sJKnBe*s6GGyGizGOEJ%$kt&F@&'
    'Nz5*CWjV#)9EUd7DgmkEh$ypkNZcQZ2UM3qS2?*U%u@8GD}+qqi?b*YYfa;4;@xg&jck)QEluvd`FQ0PXI3lBa-nNuxFNnzFY)u@Nb%4Xz-dc$0%Z_s5jp{b'
    'lE{%3&0NPg#p+5$xMc9bsxeQL=H7}H*roPG243{;$rB9i82tG~p@OqTO}flqUWbYOWxZ1o80VHPu;Q(`MM<KuA{A1Z6(^1>#U@R;U-M=oGn_XzZG#Yiz)xT`'
    '+lPXivbV5fBdoseWOwl$cERThKV(7DH=q&pJn{YL*Un3(Mgg2l3=f$}SRABkh6|tu&a3DQ0+Wx1AHv$JEXPQr?kI+d36)<~5%4HlwYsST?a(gNq3}B$?X~KC'
    'hof9*9)1qum1tv3aUYHWXNf|jC{neEhL$IXux~M9Z5HsVFulvHjvdl7kX*iewxg%x9;ZVw9eY2=RrSMYI9OWvlZX6dp4psibsA`pqgf3ES3CS+J=4%r@*HBI'
    'z1bQwmGI5e1Y-#&eAS{ZM7|hX4{OsZ;2@AGQpSB!6!hC20!d~MLmdysLazXrjr%i3?llT>vlp)3RzS??+fIP_6)LCrUY|E<vwQ^`Vuy%!A$dVeCHlkQ0K%_D'
    '`1lR4pCd9IgB}S#kK`?W-i^#C5Jf1v$#TrC5q@m)umKpJldnN<gT-E}EbZh8zE-azUbSvQ>$Qs)`7-7b@lTU8Ru{|XnD0^F0#+@iWR}`*h4=uMI;)*Grw8;M'
    '0(GG+QLS}{%u+BD*1g~hTADf}85vmVl-w-Ya%u8+G>x%skO?LvQw}yQ6#7xqc9N;{6`I6F_>?s|SSZa#qGW0#XK>R@*Y1?c6);OyN?-j1ACE(W2%6{3nmAsu'
    '5<cmLuiWvGHWAW#+Ebd3*h-Ed57!(ba%Hae<<)B+OJzsT&mF`*jOZ)9&MjGce5Q@W0bb|_LP)?Hn|T;Uw(TrKGU>dRIm)ZrpNs#Uj;w#DBiwiC$kEW&#L&gu'
    '#?;pNcPdI!U$R}{K=;Y6Hn=5&ECUwUz_lm_1Q-{C9+eu7`M`n&CRd3Evlj>|>nY!?d$n5Bv&-}P8%lK?Z??*yi-j2J#0I9g@iMKy2@$kdj$ok8HLVj0VqXC-'
    'Fh}qOhX=E;Y2{_~Sd&~73b0K32MEfn>#1>xE=C$_9gDLkn>F-s(R=M|WOjmp1=cwo$7ERI#XDJz9Y&!tLTn9+ZIm~1@D}xk-*d8O4kFPJ79jGK6b24a&(+bW'
    'e^MF&>zz#bufdnwHF5!^h)Wa^d2*FI0g77pMH+(Ca4yO?&g2CSzi!RYy4Yno>AP=f@LBInW4i|ylz@_I9D3vYYfd)(gz{fNfOQGnBXC;ZxDTi(CZMFCF*+HV'
    '23@4(y-2+h;+M0UlZMlp;?;7H83Oi@YpXYI*s*CALDk2ndh8iCP`<Cxk*|LGNv*m3j1ah))Yc!;A`}X9vxt}{(06$)%muWh#ub1Xpdg46BEj=CBNCfUg8Dv7'
    ';H2J}1C$c=tZ)5!CSYwRpqF_H*Vf*e<EN*WklCT_-M#3$;zMIAkhwX^SfV0?|FRvTI-cO^&(_4HOioTB@P&=sZf%8s5hOkVp_kC{Ksf-)1u5EwbjoGHahnRS'
    'uNgh{?3V^A-%~KYE~|_rAaQ{VZWyIk<b0a9;nOTo@RxACeNm#S16#FBHW)RzdWiLH9n26+-ay*5y%&at>vnx*!$n|NC6Iw&n;TADlH`+lWtQ<i8&Z#G1b&Y2'
    'HEWrl9;FkXpH!D@ud%xGIGmghL4P`Ml1|{tzRU#c?P2n&q)d_V&C=U>V3HKF@svX;o}l@vUG;@FaO~%GvfHJ@v^{a{0g|dKkG|nyf!EF#ei{28#@6LJdTx3n'
    'V8P;g6@=$d{>EOAkM2m^zaXO+!I54=Oud*KRNFAo3kwwa$)cATV=O-y+2JO7J_`_c#}Uf!5%aZIMY`aJ(cDeYV>ZOmAH3K|cO<>w-6fcWp)EJs0LQU0rntXB'
    '{wyF%S9<fH#!y&QlroROM+Kqj+spr+EMKz&P3+Ye4^_RCnOcv>m9)6THo%B-*@V8qh5A;>B|^)aV-abvy3b2ZWMa`PGNrQYj7n;OsJ)Oj>c$=Z^PmxPo8q)%'
    '3WTJv#cKd95|q+Yj49mG5(a3oRE}DLpMx!znxAI7*q*&U?CLzk6>#dDD^=vLc5?@d7MVZ>*r?s&nv~0ogf-1Ae@KpSn0j(!lbUGvED=Q4EA4ejs=dgfjw)=f'
    '+N&Nf4o$Bg$I96rIY1IbroHEpZGnBW#LIJ?6I7`_tSr&&TiPY9rwh+clZj)%zbbad-cwq-{j7-8b00N5J4v!$mk^GmD6JN7I+swXn4&!E9rNV(#yV=8cbT(2'
    '7@dnwp6u-uA$gle%OE!ayD^9L@u4XI!`7V_@O9v=j$>x8u(Q-Mn-0E9V1l3NZfKhN++Ve$%HavMn|{+e`;1h}Ii_ZL&{vnI?WkZ1tI4@FxiY%M;Zb^jdC6Ns'
    'wuX@ywj^Vr<@3I~RN_wtG0OwiDpT?dzjy8t0xQu59ltY~7c2+})$b#JGxcBJ$<D>m*i_%f(ALt-)XAB_%E`{QQms8+TMc>q6{9MOns0yHBOWC-UeFhyCjrx-'
    'l|ly25aCzcIFD-rPe$gs65qBzy?VV|zEDkga@ca$ex1?y>HB8XX-X#QwCmcVE0@xd%kyA(i7n;Q$?C4d+i}XLtn6qc{Z8LlTs7^2ow#ue2OPto87AEyZH*;y'
    '!&c?;42AdN9A*jtO-x*Jg06=YFb1#ZL+q5(8vLDxuc!Oo`RioU$>G2DSzZoPwl}4`y{b0Gn$N3&v?`jVMXDw2jttcIvc{3hW}gvHMTM3~0=m<Ziw0FEX8N+p'
    'eEfaAuedVTz3(UPZg)|K$9^5U_hS(jLQYtcei>&oL!8nr$3|e)IQsc)lt4}Lv$7zUjGn~AC71W*oVnI++c7Ab1L#X#ezCDdN_os`^&_c$e2MJQx{OwOI=Ixn'
    'tRx*$Bw=#(EGOH%u2p9zw2C&$N~GQaHfPB>gNGJQwiFOs7%J8<$)^htIn{+WO9ar`9<m->9xrp95;AAwK|cd~NwMH9V4W~+Ra+~IK<rrOl#gn#Nu?`w)T@j&'
    'm1N~{7p7e7xkrDsd^)p*S2br?3$8ScD=RmeV%L%=EaQ(beE8<AxsyglInTG8vnOvm{}!^DZ>2v!t*O5E6O&3dq*Kx=X+cc_QdhKW<|{_iAkk$5E~j=$4=RO&'
    '!tKq&gOW(5O3+AxS0%uzdM4+}H3_2~V24qL5u=W)G`1qXDf`i?aSr!hPDjH#dh9WjKd(>rM}eS|Zca>ORH!g*AtheE*mOm$IDTC^I>Ki`39LmTjMjc%-{rNe'
    'todKaoM1O=4P~JKv(L2Ed-d%S7$=#y1IHI_hE%G&hJ!nSF2P^~3TH6#p!R;|UaN$6Dff&CZsEyML{`(pu&9v15XHT$rjl%oGQAJsR|lO)Nm<a5nT5py=Trhk'
    'GYB2R{x-t1BE)Uc=rz=BVIZ81ieEl*i|?_wX<ei8%Z;Q#Y(Y-!_kO}a656IWo(CXM747ilb4h&u-pg+hkh{!Is|>S|yFQ*pWfn;D0zS4R|J<B%jkk0vUz;Ff'
    '-Tq^(*=c0d*L_Jst3*Ib#a)g<X9uRM+~eDKXTs8H<T+`zO)u9}8>5I>BP!ZeRv!+Pe4}sU;rkyORGmcJz3g<Da)QIkKRk+;WTI}%Ad(pdWvNh;-M|hWt~xx!'
    ';^S}a@~q-+hR@{p*C)kFxI3f~jg+tqvQOli=pb^LFu8H<J#jU#pN3t%89ZK7=`v1Fgc%z;*UZ9X*?)Hl3v(Ex|6co%hbgeCRWJtsFf76|t=?I?u%Pcg@e?Wh'
    '!ZT9Ba?cxT2L{)x=R9(ErHq8ec&rAK5{vi94|p>p0@!JFWHzm#25xDw9A6?2`O@}E#cHc)wvQ*9jobF)WpjA8@pNGmd^Lpurjd%nXB`Z6ZT2!BCm-?7UU}^G'
    '4I;PHJXmHDSyi-Is=~h+aIEmR*b5r6@sv>Xzq&-QGJihI-uUDqy(Jn!8}Tr?_{n7|5^dStLqjaqpnaaE?dph{C~*HK*q!%R%EG#byrZ*6^YfUl7UV)3`RA@r'
    'l#Nm8N=DU^V?8Jz%l8hq)2WjkqK+IP<mZdlyqQ0BJiX2lD$0O@p@aO%fd+3s5-WbUP(eXJI6?lr{6YV*HMX-gvov>NG_$ldv}XLP#$*0RQZzTww{~P==497*'
    'Ght$4V*g$E{paZSKN94>>xBQGIspa!m!14i#ozzD+cjin`fFL%-*vSIyNDY8yVd&pgYr*Z!GM8){10Wg_u)w}y!Kwd&q?j~WdZY_!h68}ON&@VOkc>^`FD+L'
    '`In{Ud&bxx9?9nK_&Bupr(g&1UkugXzfRuN`ESDSWJ9H9;6Oki$w2-(`X}5E_K%<EU*+~c4D$YlkP{gznDM)*i1i;8`X53N`a6QDyS=HS<^QeM-*aYP<SA;A'
    'KtMKr|1q%tiKCAHj`P<|fTgYZ-%z}Zon?-GQ&_lu@BI^npZ;G^{`+{wd)#}2(?2*f{{Lxr`ab%-)8L<IJmJ6d9ef}E-f-$qysG5Cw4Ztpd2gli2huI~Z;Vym'
    'W8V9J{J|(G{H+_vd%%0C?LPp@&;MR{`#tHs^z9$grq2H^g!`WMUf1ytP1)$*s64*MyjQFEgXwqp_u3WjN$;19{vic>{(meYeUE*=UgHneDCpm=-gpmsKlSzx'
    'Xe0dnMBMkW?;mLXi5*J)7mqjJBi;{C{y}i({%^yT?^*9h7yhtD3jf_$!+XyA+WrqGzU=?K(pQv$g!)Gm&hIw^Gzdsy^*{Fh50TwW4F'
)
EMBEDDED_SOURCE_SHA256 = '8dacef3c11b4bec31b83d3385610e8ba4d3fbc3c7a8fd171887a387bf55acdc4'

PROJECT = WORKING_ROOT / 'TickNets_LLargeV2CutMix200'

def project_prefixes(archive):
    names = {info.filename.replace('\\', '/') for info in archive.infolist() if not info.is_dir()}
    prefixes = []
    for name in names:
        if name == 'train_cifar_cutmix_large_v2.py' or name.endswith('/train_cifar_cutmix_large_v2.py'):
            prefix = name[:-len('train_cifar_cutmix_large_v2.py')]
            if all(prefix + relative in names for relative in SOURCE_PATHS):
                prefixes.append(prefix)
    return sorted(set(prefixes))

def find_project_source():
    if SOURCE_ZIP is not None and SOURCE_ROOT is not None:
        raise ValueError('Set only one of SOURCE_ZIP or SOURCE_ROOT')
    if SOURCE_ZIP is not None:
        candidate = Path(SOURCE_ZIP)
        if not candidate.is_file():
            raise FileNotFoundError(f'Source ZIP not found: {candidate}')
        with zipfile.ZipFile(candidate) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError(f'Source ZIP must contain one complete project; found {prefixes}')
        return 'zip', candidate, prefixes[0]
    if SOURCE_ROOT is not None:
        candidate = Path(SOURCE_ROOT)
        if not candidate.is_dir() or not all((candidate / relative).is_file() for relative in SOURCE_PATHS):
            raise FileNotFoundError(f'Source folder is incomplete: {candidate}')
        return 'directory', candidate, ''
    zip_candidates = []
    directory_candidates = []
    visited = set()
    for current, directories, files in os.walk(KAGGLE_INPUT, followlinks=True):
        resolved = Path(current).resolve()
        if resolved in visited:
            directories[:] = []
            continue
        visited.add(resolved)
        if 'train_cifar_cutmix_large_v2.py' in files and all(
                (Path(current) / relative).is_file() for relative in SOURCE_PATHS):
            directory_candidates.append(Path(current))
            directories[:] = []
            continue
        for filename in files:
            if not filename.lower().endswith('.zip'):
                continue
            candidate = Path(current) / filename
            try:
                with zipfile.ZipFile(candidate) as archive:
                    for prefix in project_prefixes(archive):
                        zip_candidates.append((candidate, prefix))
            except zipfile.BadZipFile:
                continue
    if len(zip_candidates) == 1:
        candidate, prefix = zip_candidates[0]
        return 'zip', candidate, prefix
    if not zip_candidates and len(directory_candidates) == 1:
        return 'directory', directory_candidates[0], ''
    if not zip_candidates and not directory_candidates:
        import base64, zlib
        payload = zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85))
        if hashlib.sha256(payload).hexdigest() != EMBEDDED_SOURCE_SHA256:
            raise ValueError('Embedded Phase 13 source checksum differs')
        embedded_path = WORKING_ROOT / 'TickNets_CIFAR100_LLargeV2_Embedded_Source.zip'
        embedded_path.parent.mkdir(parents=True, exist_ok=True)
        embedded_path.write_bytes(payload)
        with zipfile.ZipFile(embedded_path) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError('Embedded Phase 13 source is incomplete')
        return 'zip', embedded_path, prefixes[0]
    attached = sorted(path.name for path in Path(KAGGLE_INPUT).iterdir()) if Path(KAGGLE_INPUT).is_dir() else []
    raise RuntimeError('Could not identify one complete TickNets source in Kaggle Input. '
                       f'ZIP matches: {zip_candidates}; folder matches: {directory_candidates}; '
                       f'Input entries: {attached}. Attach the source dataset, or set '
                       'SOURCE_ZIP / SOURCE_ROOT to its exact path.')

source_type, source_path, prefix = find_project_source()
if source_type == 'zip':
    with zipfile.ZipFile(source_path) as archive:
        members = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            normalized = info.filename.replace('\\', '/')
            if normalized in members:
                raise ValueError('Source ZIP contains duplicate filenames')
            members[normalized] = info.filename
        manifest_name = prefix + 'cutmix_large_v2_200_source_manifest.json'
        manifest = json.loads(archive.read(members[manifest_name])) if manifest_name in members else None
        payloads = {relative: archive.read(members[prefix + relative]) for relative in SOURCE_PATHS}
else:
    manifest_path = source_path / 'cutmix_large_v2_200_source_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else None
    payloads = {relative: (source_path / relative).read_bytes() for relative in SOURCE_PATHS}
if manifest is not None and set(manifest) != set(SOURCE_PATHS):
    raise ValueError('Source manifest does not match the required files')
PROJECT.mkdir(parents=True, exist_ok=True)
for relative, payload in payloads.items():
    parts = PurePosixPath(relative).parts
    if not parts or any(part in ('', '.', '..') for part in parts):
        raise ValueError(f'Unsafe source path: {relative}')
    if manifest is not None and hashlib.sha256(payload).hexdigest() != manifest[relative]:
        raise ValueError(f'Source checksum differs: {relative}')
    target = (PROJECT / Path(*parts)).resolve()
    if not target.is_relative_to(PROJECT.resolve()):
        raise ValueError(f'Source path escapes project: {relative}')
    if target.exists() and target.read_bytes() != payload:
        raise RuntimeError(f'Existing extracted source differs: {target}; use a fresh session')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
missing = [module for module in ('torch', 'torchvision', 'numpy', 'PIL', 'pandas')
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f'Missing Kaggle packages: {missing}')
import torch, torchvision
assert hasattr(torchvision.transforms, 'RandAugment'), 'torchvision RandAugment is unavailable'
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running.'
print('Project source:', source_type, source_path)
print('TickNet-L Large v2 source:', hashlib.sha256((PROJECT / 'train_cifar_cutmix_large_v2.py').read_bytes()).hexdigest())
print('Runtime:', torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
import subprocess
subprocess.run([sys.executable, 'download_cifar.py', '--dataset', 'cifar100',
                '--data-root', str(WORKING_ROOT / 'data')], cwd=PROJECT, check=True)
command = [sys.executable, 'scripts/run_kaggle_cutmix_large_v2_200.py']
if EPOCHS_PER_SESSION is not None:
    command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
if RESUME_ROOT is not None:
    command += ['--resume-root', str(RESUME_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)
